In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Main Experiment Runner

This script runs the main PPS experiment using the generated participant-specific
audio files and logs mouse clicks to detect tactile stimulus responses.

Features:
- Participant selection from available design files
- Visualization of experiment timeline
- Tactile stimulus and response tracking
- Detailed analysis of participant performance
- Automatic missed trial detection and recovery
- LSL markers for clicks and stimuli
- Real-time audio generation for missed trials

Author: AI Assistant
"""

import os
import time
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext, filedialog
import sounddevice as sd
import soundfile as sf
import numpy as np
import pandas as pd
import threading
import random
import datetime
import json
import re
import glob
from pathlib import Path
import traceback
import wave
import struct
import io
import shutil
from pydub import AudioSegment

# Import LSL library for marker streaming
try:
    from pylsl import StreamInfo, StreamOutlet
    LSL_AVAILABLE = True
except ImportError:
    print("WARNING: pylsl not available. LSL markers will be disabled.")
    LSL_AVAILABLE = False

# Configuration
BASE_DIR = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace"
EXPERIMENT_AUDIO_DIR = os.path.join(BASE_DIR, "Level1_AudioGeneration", "ExperimentAudio")
EXPERIMENT_LOG_DIR = os.path.join(BASE_DIR, "Level1_AudioGeneration", "ExperimentLog")
STIMULUS_DIR = os.path.join(BASE_DIR, "Level0_StimulusFiles")
RESULTS_DIR = os.path.join(BASE_DIR, "Level2_RunExperiment", "Results")
RECOVERY_DIR = os.path.join(BASE_DIR, "Level2_RunExperiment", "Recovery")

# Response window in seconds
RESPONSE_WINDOW = 1.5

# Audio parameters
SAMPLE_RATE = 48000
AUDIO_CHANNELS = 2

# LSL configuration
LSL_MOUSE_STREAM_NAME = "PPSMouseClicks"
LSL_TACTILE_STREAM_NAME = "PPSTactileMarkers"
LSL_AUDIO_STREAM_NAME = "PPSAudioMarkers"

# Ensure directories exist
for dir_path in [RESULTS_DIR, RECOVERY_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# Dictionary mapping stimulus types to filenames
LOOMING_STIMULUS_FILES = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav',
}

class PPSExperimentRunner:
    def __init__(self):
        # Initialize variables
        self.participant_id = None
        self.available_participants = []
        self.experiment_running = False
        self.start_time = None
        self.mouse_clicks = []
        self.current_log_file = None
        self.tactile_times = []
        self.timeline_markers = []
        self.click_count = 0
        self.recovery_mode = False
        self.missed_trials = []
        self.original_design_df = None
        self.trial_responses = {}  # Track responses to each trial
        self.recovery_generated = False
        self.recovery_audio_files = None
        self.recovery_tactile_times = []
        
        # Flag to control audio playback
        self.stop_audio = False
        
        # Setup LSL outlets
        self.setup_lsl_outlets()
        
        # Scan for available participants
        self.scan_available_participants()
        
        # Create GUI
        self.create_gui()
    
    def setup_lsl_outlets(self):
        """Initialize LSL stream outlets for markers."""
        self.mouse_click_outlet = None
        self.tactile_marker_outlet = None
        self.audio_marker_outlet = None
        
        if LSL_AVAILABLE:
            try:
                # Create stream info for mouse clicks
                mouse_info = StreamInfo(
                    name=LSL_MOUSE_STREAM_NAME,
                    type='Markers',
                    channel_count=1,
                    nominal_srate=0,  # Irregular sampling rate
                    channel_format='string',
                    source_id='pps_experiment_mouse'
                )
                
                # Create stream info for tactile markers
                tactile_info = StreamInfo(
                    name=LSL_TACTILE_STREAM_NAME,
                    type='Markers',
                    channel_count=1,
                    nominal_srate=0,  # Irregular sampling rate
                    channel_format='string',
                    source_id='pps_experiment_tactile'
                )
                
                # Create stream info for audio playback markers
                audio_info = StreamInfo(
                    name=LSL_AUDIO_STREAM_NAME,
                    type='Markers',
                    channel_count=1,
                    nominal_srate=0,  # Irregular sampling rate
                    channel_format='string',
                    source_id='pps_experiment_audio'
                )
                
                # Add some metadata
                mouse_info.desc().append_child_value("manufacturer", "PPSExperiment")
                tactile_info.desc().append_child_value("manufacturer", "PPSExperiment")
                audio_info.desc().append_child_value("manufacturer", "PPSExperiment")
                
                # Create outlets
                self.mouse_click_outlet = StreamOutlet(mouse_info)
                self.tactile_marker_outlet = StreamOutlet(tactile_info)
                self.audio_marker_outlet = StreamOutlet(audio_info)
                
                print("LSL streams initialized successfully")
            except Exception as e:
                print(f"Error initializing LSL streams: {e}")
                traceback.print_exc()
        else:
            print("LSL not available - markers will not be sent")

    def scan_available_participants(self):
        """Scan for available participants based on design files."""
        self.available_participants = []
        
        # Look for participant design files
        pattern = os.path.join(EXPERIMENT_LOG_DIR, "participant_*_design.csv")
        
        for file_path in glob.glob(pattern):
            # Extract participant ID from filename
            match = re.search(r'participant_(\d+)_design\.csv', os.path.basename(file_path))
            if match:
                participant_id = int(match.group(1))
                
                # Check if corresponding audio files exist
                looming_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{participant_id}_design_looming.wav")
                tactile_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{participant_id}_design_tactile.wav")
                
                if os.path.exists(looming_file) and os.path.exists(tactile_file):
                    self.available_participants.append(participant_id)
        
        # Sort numerically
        self.available_participants.sort()
        
        print(f"Found {len(self.available_participants)} available participants")
        
    def update_progress_line(self, elapsed_time):
        """Update the position of the progress line based on elapsed time."""
        try:
            # Calculate position
            timeline_start_x = 400
            timeline_width = self.timeline_end_x - timeline_start_x
            max_duration = 600  # Maximum duration in seconds (10 minutes)
            
            x_pos = timeline_start_x + min(elapsed_time / max_duration, 1.0) * timeline_width
            
            # Update line position
            self.visual_canvas.itemconfig(self.progress_line, state="normal")
            self.visual_canvas.coords(
                self.progress_line, 
                x_pos, self.timeline_y - 30, 
                x_pos, self.timeline_y + 30
            )
            return True
        except Exception as e:
            print(f"Error updating progress line: {e}")
            return False
    
    def create_gui(self):
        """Create the GUI."""
        self.root = tk.Tk()
        self.root.title("PPS Main Experiment")
        
        # Set protocol for window close
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)
        
        # Get screen dimensions
        screen_width = self.root.winfo_screenwidth()
        screen_height = self.root.winfo_screenheight()
        
        # Set window to 90% of screen size to account for taskbar and window decorations
        window_width = int(screen_width * 0.9)
        window_height = int(screen_height * 0.9)
        self.root.geometry(f"{window_width}x{window_height}+{int(screen_width*0.05)}+{int(screen_height*0.05)}")
        
        # Allow window resizing
        self.root.resizable(True, True)
        
        # Create main canvas with scrollbar
        self.main_canvas = tk.Canvas(self.root)
        self.main_canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        # Add vertical scrollbar to canvas
        self.vsb = ttk.Scrollbar(self.root, orient=tk.VERTICAL, command=self.main_canvas.yview)
        self.vsb.pack(side=tk.RIGHT, fill=tk.Y)
        self.main_canvas.configure(yscrollcommand=self.vsb.set)
        
        # Create a frame inside the canvas for all content
        self.main_frame = ttk.Frame(self.main_canvas, padding="10")
        self.main_canvas.create_window((0, 0), window=self.main_frame, anchor=tk.NW, tags="self.main_frame")
        
        # Configure the canvas to resize with the frame
        self.main_frame.bind("<Configure>", lambda e: self.main_canvas.configure(scrollregion=self.main_canvas.bbox("all")))
        
        # Bind mousewheel to scroll
        self.root.bind("<MouseWheel>", lambda e: self.main_canvas.yview_scroll(int(-1*(e.delta/120)), "units"))  # For Windows
        self.root.bind("<Button-4>", lambda e: self.main_canvas.yview_scroll(-1, "units"))  # For Linux
        self.root.bind("<Button-5>", lambda e: self.main_canvas.yview_scroll(1, "units"))   # For Linux
        
        # Initialize all widgets that will be referenced later
        self.finish_button = None
        self.show_diagnostic_button = None
        self.start_button = None
        self.timeline_markers = []
        
        # Bind mouse clicks for the entire application - do this FIRST
        self.root.bind("<Button-1>", self.on_mouse_click)
        print("Mouse click binding established")
        
        # Header
        ttk.Label(self.main_frame, text="PPS Main Experiment", 
                  font=("Arial", 18, "bold")).pack(pady=5)
        
        # Participant selection frame
        participant_frame = ttk.LabelFrame(self.main_frame, text="Participant Selection", padding=10)
        participant_frame.pack(fill=tk.X, padx=5, pady=5)
        
        # Horizontal frame for participant selection
        participant_selection_frame = ttk.Frame(participant_frame)
        participant_selection_frame.pack(fill=tk.X, pady=5)
        
        # Participant ID selection
        ttk.Label(participant_selection_frame, text="Participant ID:").pack(side=tk.LEFT, padx=5)
        
        self.participant_var = tk.StringVar()
        if self.available_participants:
            self.participant_var.set(str(self.available_participants[0]))
        
        participant_dropdown = ttk.Combobox(
            participant_selection_frame, 
            textvariable=self.participant_var,
            values=[str(p) for p in self.available_participants],
            width=5
        )
        participant_dropdown.pack(side=tk.LEFT, padx=5)
        
        # Refresh button
        ttk.Button(
            participant_selection_frame, 
            text="Refresh List",
            command=self.refresh_participants
        ).pack(side=tk.LEFT, padx=10)
        
        # Recovery mode checkbox
        self.recovery_var = tk.BooleanVar(value=False)
        self.recovery_checkbox = ttk.Checkbutton(
            participant_selection_frame,
            text="Recovery Mode (Missed Trials)",
            variable=self.recovery_var,
            state=tk.DISABLED
        )
        self.recovery_checkbox.pack(side=tk.LEFT, padx=20)
        
        # Instructions frame
        instructions_frame = ttk.LabelFrame(self.main_frame, text="Instructions", padding=5)
        instructions_frame.pack(fill=tk.X, padx=5, pady=5)
        
        instruction_text = (
            "1. Select a participant ID from the dropdown\n"
            "2. Click 'Start Experiment' to begin\n"
            "3. Instruct the participant to click the mouse button ONLY when they hear a tactile stimulus\n"
            f"4. Participant must click within {RESPONSE_WINDOW} seconds after hearing the tactile stimulus\n"
            "5. Participant should NOT click when they only hear a looming sound without tactile stimulus\n"
            "6. The experiment will automatically complete when the audio concludes"
        )
        
        ttk.Label(instructions_frame, text=instruction_text, 
                  font=("Arial", 10), justify="left").pack(anchor=tk.W, pady=2)
        
        # Status frame
        self.status_frame = ttk.Frame(self.main_frame)
        self.status_frame.pack(fill=tk.X, padx=5, pady=2)
        
        self.status_var = tk.StringVar(value="Select a participant to begin")
        self.status_label = ttk.Label(self.status_frame, textvariable=self.status_var, 
                                      font=("Arial", 11, "bold"))
        self.status_label.pack(pady=2)
        
        # Progress bar frame
        progress_frame = ttk.Frame(self.main_frame)
        progress_frame.pack(fill=tk.X, padx=5, pady=5)
        
        self.progress_var = tk.DoubleVar(value=0.0)
        self.progress_bar = ttk.Progressbar(
            progress_frame, 
            orient="horizontal", 
            length=window_width-50,
            mode="determinate",
            variable=self.progress_var
        )
        self.progress_bar.pack(fill=tk.X, pady=5)
        
        # Experimenter visualization frame
        visual_frame = ttk.LabelFrame(self.main_frame, text="Experimenter Visualization", padding=5)
        visual_frame.pack(fill=tk.X, padx=5, pady=2)
        
        # Create canvas for visual indicators
        self.visual_canvas = tk.Canvas(visual_frame, width=window_width - 100, height=160, bg="white")
        self.visual_canvas.pack(fill=tk.X, pady=2)
        
        # Create indicators
        self.tactile_indicator = self.visual_canvas.create_rectangle(50, 20, 350, 45, fill="lightgray", outline="black")
        self.visual_canvas.create_text(200, 32, text="Tactile Stimulus", font=("Arial", 9))
        
        self.click_indicator = self.visual_canvas.create_rectangle(50, 55, 350, 80, fill="lightgray", outline="black")
        self.visual_canvas.create_text(200, 67, text="Mouse Click", font=("Arial", 9))
        
        # Add counters display
        self.click_counter_text = self.visual_canvas.create_text(
            650, 32,
            text="Total Clicks: 0",
            font=("Arial", 11, "bold"),
            fill="blue"
        )
        
        self.trials_counter_text = self.visual_canvas.create_text(
            650, 67,
            text="Tactile Events: 0",
            font=("Arial", 11, "bold"),
            fill="green"
        )
        
        # Timeline - make it span almost the entire width
        timeline_start_x = 50
        self.timeline_end_x = window_width - 120
        self.timeline_y = 120
        self.visual_canvas.create_line(timeline_start_x, self.timeline_y, self.timeline_end_x, self.timeline_y, width=2)
        
        # Add labels to the timeline
        self.visual_canvas.create_text(timeline_start_x - 30, self.timeline_y, text="Timeline:", font=("Arial", 9, "bold"))
        
        # Add time markers every minute (60 seconds)
        max_duration = 600  # 10 minutes
        for sec in range(0, max_duration + 1, 60):
            x_pos = timeline_start_x + ((self.timeline_end_x - timeline_start_x) * sec / max_duration)
            self.visual_canvas.create_line(x_pos, self.timeline_y - 5, x_pos, self.timeline_y + 5, width=1)
            self.visual_canvas.create_text(x_pos, self.timeline_y + 15, text=f"{sec//60}m", font=("Arial", 8))
        
        # Create progress indicator (vertical line) - represents current playback position
        self.progress_line = self.visual_canvas.create_line(
            timeline_start_x, self.timeline_y - 20, timeline_start_x, self.timeline_y + 20, 
            width=3, fill="green", state="hidden"
        )
        
        # Results frame (initially hidden)
        self.results_frame = ttk.LabelFrame(self.main_frame, text="Results", padding=5)
        self.results_var = tk.StringVar(value="Run the experiment to see results")
        self.results_label = ttk.Label(self.results_frame, textvariable=self.results_var, 
                                      font=("Arial", 10), justify="left")
        self.results_label.pack(anchor=tk.W, pady=2)
        # Don't pack the results frame yet - it will be shown after experiment
        
        # Button styles
        self.button_style = ttk.Style()
        self.button_style.configure("Start.TButton", font=("Arial", 12, "bold"))
        self.button_style.configure("Green.TButton", background="green")
        
        # Create a distinctive frame for the Start button
        start_frame = ttk.LabelFrame(self.main_frame, text="Experiment Control", padding=5)
        start_frame.pack(fill=tk.X, padx=5, pady=5)
        
        # Create a horizontal frame for the buttons
        buttons_horizontal = ttk.Frame(start_frame)
        buttons_horizontal.pack(fill=tk.X, pady=5)
        
        # Start button
        self.start_button = ttk.Button(
            buttons_horizontal, text="START EXPERIMENT",
            command=self.start_experiment, width=25, style="Start.TButton"
        )
        self.start_button.pack(side=tk.LEFT, padx=5)
        
        # Add Diagnostic Report button (hidden initially)
        self.show_diagnostic_button = ttk.Button(
            buttons_horizontal, text="Show Detailed Diagnostics",
            command=self.show_diagnostic_report, state=tk.DISABLED
        )
        self.show_diagnostic_button.pack(side=tk.LEFT, padx=5)
        self.show_diagnostic_button.pack_forget()  # Hide initially
        
        # Cancel button
        self.cancel_button = ttk.Button(
            buttons_horizontal, text="Cancel",
            command=self.on_close, width=15
        )
        self.cancel_button.pack(side=tk.RIGHT, padx=5)
        
        # Finish button (hidden initially)
        self.finish_button = ttk.Button(
            buttons_horizontal, text="Finish & Save Results",
            command=self.finish_experiment, width=20, state=tk.DISABLED
        )
        self.finish_button.pack(side=tk.RIGHT, padx=5)
        self.finish_button.pack_forget()  # Hide initially
        
        # Ensure scrolling works properly
        self.main_frame.update_idletasks()
        self.main_canvas.config(scrollregion=self.main_canvas.bbox("all"))
        
        # Scroll to top when starting
        self.main_canvas.yview_moveto(0)
    
    def refresh_participants(self):
        """Refresh the list of available participants."""
        previous_selection = self.participant_var.get()
        
        # Scan for available participants
        self.scan_available_participants()
        
        # Update dropdown values
        participant_dropdown = self.root.nametowidget(
            self.participant_var.winfo_pathname(self.participant_var.winfo_id())
        )
        participant_dropdown['values'] = [str(p) for p in self.available_participants]
        
        # Try to keep previous selection if it still exists
        if previous_selection in [str(p) for p in self.available_participants]:
            self.participant_var.set(previous_selection)
        elif self.available_participants:
            self.participant_var.set(str(self.available_participants[0]))
        else:
            self.participant_var.set("")
        
        # Check for recovery data for this participant
        self.check_recovery_data()
    
    def check_recovery_data(self):
        """Check if recovery data exists for the selected participant."""
        try:
            participant_id = int(self.participant_var.get())
            results_file = os.path.join(RESULTS_DIR, f"participant_{participant_id}_results.json")
            
            if os.path.exists(results_file):
                # Load results to check for missed trials
                with open(results_file, 'r') as f:
                    results = json.load(f)
                
                if 'missed_trials' in results and results['missed_trials']:
                    # Enable recovery mode
                    self.recovery_checkbox.config(state=tk.NORMAL)
                    self.missed_trials = results['missed_trials']
                    print(f"Found {len(self.missed_trials)} missed trials for recovery")
                    return
            
            # No recovery data found
            self.recovery_checkbox.config(state=tk.DISABLED)
            self.recovery_var.set(False)
            self.missed_trials = []
            
        except Exception as e:
            print(f"Error checking recovery data: {e}")
            self.recovery_checkbox.config(state=tk.DISABLED)
            self.recovery_var.set(False)
    
    def show_diagnostic_report(self):
        """Show detailed diagnostic report in a new window."""
        if not hasattr(self, 'diagnostic_report'):
            return
            
        # Create new window for report that fits screen better
        report_window = tk.Toplevel(self.root)
        report_window.title("Diagnostic Report")
        
        # Set size relative to screen
        window_width = min(800, self.root.winfo_screenwidth() - 100)
        window_height = min(600, self.root.winfo_screenheight() - 100)
        report_window.geometry(f"{window_width}x{window_height}")
        
        # Add scrolled text widget instead of manual scrollbar
        text_widget = scrolledtext.ScrolledText(
            report_window, wrap=tk.WORD, width=80, height=25
        )
        text_widget.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Insert report text
        text_widget.insert(tk.END, self.diagnostic_report)
        text_widget.config(state=tk.DISABLED)  # Make read-only
    
    def on_close(self):
        """Handle window close event."""
        if self.experiment_running:
            if messagebox.askyesno("Quit", "Experiment is running. Are you sure you want to quit?"):
                # Stop audio playback when closing the window
                self.stop_audio = True
                sd.stop()
                print("Audio playback stopped")
                self.root.destroy()
        else:
            self.root.destroy()
    
    def on_mouse_click(self, event):
        """Handle mouse click events during the experiment."""
        # Ignore clicks in non-experiment phase
        if not self.experiment_running or self.start_time is None:
            return
            
        # Calculate time since experiment start
        current_time = time.perf_counter() - self.start_time
        
        # Create a timestamp for LSL
        timestamp_iso = datetime.datetime.now().isoformat()
        
        # Send LSL marker for mouse click
        if self.mouse_click_outlet is not None:
            try:
                marker_str = f"CLICK,{current_time:.6f},{self.participant_id}"
                self.mouse_click_outlet.push_sample([marker_str])
                print(f"Sent LSL marker: {marker_str}")
            except Exception as e:
                print(f"Error sending LSL marker: {e}")
        
        # Add to mouse clicks list
        self.mouse_clicks.append({
            "time": current_time,
            "timestamp": timestamp_iso,
            "x": event.x,
            "y": event.y
        })
        
        # Update click count
        self.click_count += 1
        try:
            self.visual_canvas.itemconfig(
                self.click_counter_text, 
                text=f"Total Clicks: {self.click_count}"
            )
        except Exception as e:
            print(f"Error updating click counter: {e}")
        
        # Visual indicator for the experimenter
        self.flash_indicator(self.click_indicator, "green")
        
        # Process the click for reaction time analysis
        self.process_click_for_reaction_time(current_time)
        
        # Add click marker to timeline
        try:
            self.add_timeline_marker(current_time, "red")
        except Exception as e:
            print(f"Error adding timeline marker: {e}")
    
    def process_click_for_reaction_time(self, click_time):
        """
        Process a click to determine if it's a valid response to a tactile stimulus
        and calculate reaction time.
        """
        # Find the closest preceding tactile stimulus
        closest_idx = -1
        closest_diff = float('inf')
        closest_trial_num = None
        
        # Iterate through tactile times to find the closest one
        if hasattr(self, 'tactile_times_data') and self.tactile_times_data:
            # Using enhanced tactile data structure
            for i, tactile_data in enumerate(self.tactile_times_data):
                time_diff = click_time - tactile_data['time']
                if 0 < time_diff <= RESPONSE_WINDOW and time_diff < closest_diff:
                    closest_diff = time_diff
                    closest_idx = i
                    closest_trial_num = tactile_data['trial_number']
        else:
            # Fallback to simple tactile times list
            for i, stim_time in enumerate(self.tactile_times):
                time_diff = click_time - stim_time
                if 0 < time_diff <= RESPONSE_WINDOW and time_diff < closest_diff:
                    closest_diff = time_diff
                    closest_idx = i
        
        # If we found a matching tactile stimulus
        if closest_idx >= 0:
            print(f"  ✓ Valid response: Click at {click_time:.3f}s matches tactile at {self.tactile_times[closest_idx]:.3f}s (RT: {closest_diff:.3f}s)")
            
            # If we have the trial number, update the response tracking
            if closest_trial_num is not None and closest_trial_num in self.trial_responses:
                self.trial_responses[closest_trial_num]['responded'] = True
                self.trial_responses[closest_trial_num]['reaction_time'] = closest_diff
                print(f"  ✓ Marked trial {closest_trial_num} as responded with RT: {closest_diff:.3f}s")
        else:
            print(f"  ✗ Invalid response: Click at {click_time:.3f}s doesn't match any tactile stimulus within {RESPONSE_WINDOW}s window")
    
    def flash_indicator(self, indicator, color, duration=0.5):
        """Flash an indicator on the canvas for the experimenter."""
        try:
            original_color = self.visual_canvas.itemcget(indicator, "fill")
            self.visual_canvas.itemconfig(indicator, fill=color)
            self.root.update_idletasks()
            
            # Schedule color change back
            def reset_color():
                try:
                    if self.visual_canvas.winfo_exists():
                        self.visual_canvas.itemconfig(indicator, fill=original_color)
                except Exception as e:
                    print(f"Error resetting indicator color: {e}")
            
            self.root.after(int(duration * 1000), reset_color)
            return True
        except Exception as e:
            print(f"Error flashing indicator: {e}")
            return False
    
    def add_timeline_marker(self, time_sec, color):
        """Add a marker to the timeline at the specified time."""
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        # Calculate position based on time
        max_duration = 600  # Maximum duration in seconds (10 minutes)
        x_pos = timeline_start_x + min(time_sec / max_duration, 1.0) * timeline_width
        
        try:
            # Create marker
            marker = self.visual_canvas.create_oval(
                x_pos-5, self.timeline_y-5, x_pos+5, self.timeline_y+5, 
                fill=color, outline="black", width=1
            )
            self.timeline_markers.append(marker)
            
            # Update progress line
            self.visual_canvas.itemconfig(self.progress_line, state="normal")
            self.visual_canvas.coords(
                self.progress_line, 
                x_pos, self.timeline_y-20, 
                x_pos, self.timeline_y+20
            )
            
            return True
        except Exception as e:
            print(f"Error creating timeline marker: {e}")
            return False
    
    def clear_timeline(self):
        """Clear all markers from the timeline."""
        print("Clearing timeline markers...")
        try:
            for marker in self.timeline_markers:
                self.visual_canvas.delete(marker)
            self.timeline_markers = []
            
            print("Timeline cleared successfully")
        except Exception as e:
            print(f"Error clearing timeline: {e}")
    
    def start_experiment(self):
        """Start the main experiment."""
        # Get participant ID
        try:
            self.participant_id = int(self.participant_var.get())
        except (ValueError, TypeError):
            messagebox.showerror("Error", "Please select a valid participant ID")
            return
        
        self.recovery_mode = self.recovery_var.get()
        
        print(f"\n===== STARTING {'RECOVERY' if self.recovery_mode else 'MAIN'} EXPERIMENT =====")
        print(f"Participant ID: {self.participant_id}")
        print(f"Recovery Mode: {self.recovery_mode}")
        
        # Reset stop audio flag
        self.stop_audio = False
        
        # Reset click counter
        self.click_count = 0
        self.visual_canvas.itemconfig(self.click_counter_text, text=f"Total Clicks: 0")
        
        # Disable controls during experiment
        self.start_button.config(state=tk.DISABLED)
        self.recovery_checkbox.config(state=tk.DISABLED)
        
        # Reset variables
        self.mouse_clicks = []
        self.experiment_running = True
        
        # Reset indicators
        self.visual_canvas.itemconfig(self.tactile_indicator, fill="lightgray")
        self.visual_canvas.itemconfig(self.click_indicator, fill="lightgray")
        
        # Hide progress line initially
        self.visual_canvas.itemconfig(self.progress_line, state="hidden")
        
        # Update status
        self.status_var.set(f"Starting experiment for participant {self.participant_id}...")
        self.root.update()
        
        # Create log file
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        phase = "recovery" if self.recovery_mode else "main"
        self.current_log_file = os.path.join(RESULTS_DIR, f"participant_{self.participant_id}_{phase}_log_{timestamp}.csv")
        
        # Clear previous timeline markers
        self.clear_timeline()
        
        # Start experiment in separate thread
        threading.Thread(target=self.run_experiment, daemon=True).start()
    
    def load_trial_data(self):
        """Load the trial data for the selected participant."""
        try:
            if self.recovery_mode and self.recovery_audio_files:
                # Use recovery tactile times
                print("Using generated recovery tactile times")
                return self.recovery_tactile_times
            elif self.recovery_mode:
                # Use missed trials as trial data
                print("Using missed trials for recovery session")
                tactile_times = []
                for trial in self.missed_trials:
                    if 'tactile_time' in trial and trial['tactile_time'] is not None:
                        tactile_times.append(trial['tactile_time'])
                
                print(f"Loaded {len(tactile_times)} tactile times for recovery")
                return tactile_times
            else:
                # Load from original design file
                design_file = os.path.join(EXPERIMENT_LOG_DIR, f"participant_{self.participant_id}_design.csv")
                print(f"Loading trial data from: {design_file}")
                
                self.original_design_df = pd.read_csv(design_file)
                print(f"Loaded design with {len(self.original_design_df)} trials")
                
                # Initialize response tracking for all trials
                for i, row in self.original_design_df.iterrows():
                    trial_num = row['trial_number']
                    self.trial_responses[trial_num] = {
                        'trial_number': trial_num,
                        'trial_type': row['trial_type'],
                        'responded': False,
                        'reaction_time': None,
                        'trial_data': row.to_dict()
                    }
                
                # Filter for trials with tactile stimuli (exclude catch trials)
                tactile_trials = self.original_design_df[self.original_design_df['trial_type'] != 'catch']
                
                # Extract tactile stimulus times
                tactile_times = []
                for _, row in tactile_trials.iterrows():
                    if 'tactile_stimulus_timestamp' in row and pd.notna(row['tactile_stimulus_timestamp']):
                        ts_str = row['tactile_stimulus_timestamp']
                        # Parse timestamp format MM:SS.S
                        match = re.match(r'(\d+):(\d+\.\d+)', ts_str)
                        if match:
                            minutes, seconds = match.groups()
                            time_sec = float(minutes) * 60 + float(seconds)
                            # Store trial number with each tactile time for later reference
                            tactile_times.append({
                                'time': time_sec,
                                'trial_number': row['trial_number']
                            })
                
                # Extract just the times for backward compatibility
                tactile_times_simple = [t['time'] for t in tactile_times]
                
                print(f"Loaded {len(tactile_times)} tactile stimulus times")
                return tactile_times_simple
        except Exception as e:
            print(f"Error loading trial data: {e}")
            traceback.print_exc()
            return []
    
    def parse_timestamp(self, timestamp_str):
        """Parse timestamp in format MM:SS.S to seconds."""
        if pd.isna(timestamp_str):
            return None
            
        match = re.match(r'(\d+):(\d+\.\d+)', timestamp_str)
        if match:
            minutes, seconds = match.groups()
            return float(minutes) * 60 + float(seconds)
        return None
    
    def run_experiment(self):
        """Run the actual experiment."""
        try:
            # Get file paths
            phase = "recovery" if self.recovery_mode else "main"
            
            # Check if we're using recovery audio that was generated on-the-fly
            if self.recovery_mode and self.recovery_audio_files:
                looming_file = self.recovery_audio_files['looming_path']
                tactile_file = self.recovery_audio_files['tactile_path']
            else:
                looming_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{self.participant_id}_design_looming.wav")
                tactile_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{self.participant_id}_design_tactile.wav")
            
            print(f"Using audio files:")
            print(f"- Looming: {looming_file}")
            print(f"- Tactile: {tactile_file}")
            
            # Check if files exist
            if not os.path.exists(looming_file) or not os.path.exists(tactile_file):
                self.status_var.set(f"Error: Audio files for participant {self.participant_id} not found")
                print(f"ERROR: Could not find one or more audio files")
                self.experiment_running = False
                self.start_button.config(state=tk.NORMAL)
                return
            
            # Load tactile stimulus times
            self.tactile_times = self.load_trial_data()
            if not self.tactile_times:
                self.status_var.set(f"Error: No tactile stimulus times found for participant {self.participant_id}")
                print(f"ERROR: No tactile stimulus times found")
                self.experiment_running = False
                self.start_button.config(state=tk.NORMAL)
                return
            
            # Store enhanced tactile times data if we have trial information
            self.tactile_times_data = []
            if hasattr(self, 'original_design_df') and self.original_design_df is not None:
                for i, t_time in enumerate(self.tactile_times):
                    # Try to find the corresponding trial number
                    matching_rows = self.original_design_df[
                        (self.original_design_df['trial_type'] != 'catch') & 
                        (self.original_design_df['tactile_stimulus_timestamp'].notna())
                    ]
                    
                    for _, row in matching_rows.iterrows():
                        ts_time = self.parse_timestamp(row['tactile_stimulus_timestamp'])
                        if ts_time is not None and abs(ts_time - t_time) < 0.1:  # Close match
                            self.tactile_times_data.append({
                                'time': t_time,
                                'trial_number': row['trial_number'],
                                'trial_type': row['trial_type'],
                                'stimulus_type': row.get('stimulus_type', None)
                            })
                            break
                    else:
                        # No match found, add just the time
                        self.tactile_times_data.append({
                            'time': t_time,
                            'trial_number': i + 1,
                            'trial_type': 'unknown',
                            'stimulus_type': None
                        })
            
            # Update trials counter
            self.visual_canvas.itemconfig(
                self.trials_counter_text, 
                text=f"Tactile Events: {len(self.tactile_times)}"
            )
            
            # Add tactile event markers to timeline
            for t_time in self.tactile_times:
                try:
                    self.add_timeline_marker(t_time, "blue")
                except Exception as e:
                    print(f"Error adding tactile marker: {e}")
            
            # Load audio files
            looming_data, looming_sr = sf.read(looming_file)
            tactile_data, tactile_sr = sf.read(tactile_file)
            
            print(f"Loaded audio files:")
            print(f"- Looming: {len(looming_data)/looming_sr:.2f}s, {looming_sr}Hz, shape: {looming_data.shape}")
            print(f"- Tactile: {len(tactile_data)/tactile_sr:.2f}s, {tactile_sr}Hz, shape: {tactile_data.shape}")
            
            # Verify both audio files have the same sample rate and length
            if looming_sr != tactile_sr:
                print(f"WARNING: Sample rate mismatch between looming ({looming_sr}Hz) and tactile ({tactile_sr}Hz)")
            
            if len(looming_data) != len(tactile_data):
                print(f"WARNING: Length mismatch between looming ({len(looming_data)}) and tactile ({len(tactile_data)})")
            
            # Convert stereo to mono if needed
            if len(looming_data.shape) > 1 and looming_data.shape[1] > 1:
                print("Converting looming audio from stereo to mono for playback")
                looming_data = np.mean(looming_data, axis=1)
            if len(tactile_data.shape) > 1 and tactile_data.shape[1] > 1:
                print("Converting tactile audio from stereo to mono for playback")
                tactile_data = np.mean(tactile_data, axis=1)
            
            def update_status(msg):
                try:
                    self.status_var.set(msg)
                    self.root.update_idletasks()
                except Exception as e:
                    print(f"Error updating status: {e}")
            
            self.root.after(0, update_status, f"Experiment running - participant should click when they hear a tactile stimulus")
            
            # Set start time just before playback
            self.start_time = time.perf_counter()
            print(f"Starting audio playback at {datetime.datetime.now().strftime('%H:%M:%S.%f')}")
            
            # Send LSL marker for experiment start
            if self.audio_marker_outlet is not None:
                try:
                    phase_name = "RECOVERY" if self.recovery_mode else "MAIN"
                    marker_str = f"AUDIO_START,{phase_name},{self.participant_id},0.0"
                    self.audio_marker_outlet.push_sample([marker_str])
                    print(f"Sent LSL marker: {marker_str}")
                except Exception as e:
                    print(f"Error sending LSL marker: {e}")
            
            # Reset progress line to start position
            self.update_progress_line(0)
            
            # Start monitor thread for tactile indicators
            tactile_monitor_thread = threading.Thread(
                target=self.monitor_tactile_events, 
                daemon=True
            )
            tactile_monitor_thread.start()
            
            # Show the finish button (but keep it disabled until experiment finishes)
            self.finish_button.pack(side=tk.RIGHT, padx=5)
            
            # Calculate audio duration
            max_duration = len(looming_data) / looming_sr
            
            # Play both audio files in separate threads with checks for the stop flag
            def play_looming():
                try:
                    # Send LSL marker for looming audio start
                    if self.audio_marker_outlet is not None:
                        try:
                            phase_name = "RECOVERY" if self.recovery_mode else "MAIN"
                            marker_str = f"LOOMING_START,{phase_name},{self.participant_id},0.0"
                            self.audio_marker_outlet.push_sample([marker_str])
                            print(f"Sent LSL marker: {marker_str}")
                        except Exception as e:
                            print(f"Error sending LSL marker: {e}")
                    
                    # Play main audio
                    with sd.OutputStream(samplerate=looming_sr, channels=1, device=0) as stream:
                        chunk_size = 1024
                        for i in range(0, len(looming_data), chunk_size):
                            if self.stop_audio:
                                print("Looming audio playback stopped")
                                break
                            
                            # Send marker every 30 seconds for synchronization
                            if i % (30 * looming_sr) < chunk_size and i > 0:
                                current_time = (i / looming_sr)
                                if self.audio_marker_outlet is not None:
                                    try:
                                        marker_str = f"LOOMING_SYNC,{self.participant_id},{current_time:.2f}"
                                        self.audio_marker_outlet.push_sample([marker_str])
                                    except Exception as e:
                                        print(f"Error sending sync marker: {e}")
                            
                            chunk = looming_data[i:min(i+chunk_size, len(looming_data))]
                            stream.write(chunk.astype(np.float32))
                    
                    # Send LSL marker for looming audio end
                    if self.audio_marker_outlet is not None:
                        try:
                            phase_name = "RECOVERY" if self.recovery_mode else "MAIN"
                            current_time = time.perf_counter() - self.start_time
                            marker_str = f"LOOMING_END,{phase_name},{self.participant_id},{current_time:.2f}"
                            self.audio_marker_outlet.push_sample([marker_str])
                            print(f"Sent LSL marker: {marker_str}")
                        except Exception as e:
                            print(f"Error sending LSL marker: {e}")
                    
                    print("Looming audio playback completed")
                    
                    # If recovery audio exists and we're not already in recovery mode, play that next
                    if self.recovery_audio_files and not self.recovery_mode and not self.stop_audio:
                        print("Playing recovery audio sequence for missed trials")
                        
                        # Send LSL marker for recovery looming start
                        if self.audio_marker_outlet is not None:
                            try:
                                current_time = time.perf_counter() - self.start_time
                                marker_str = f"RECOVERY_LOOMING_START,{self.participant_id},{current_time:.2f}"
                                self.audio_marker_outlet.push_sample([marker_str])
                                print(f"Sent LSL marker: {marker_str}")
                            except Exception as e:
                                print(f"Error sending LSL marker: {e}")
                        
                        recovery_looming, recovery_sr = sf.read(self.recovery_audio_files['looming_path'])
                        with sd.OutputStream(samplerate=recovery_sr, channels=1, device=0) as stream:
                            chunk_size = 1024
                            for i in range(0, len(recovery_looming), chunk_size):
                                if self.stop_audio:
                                    print("Recovery looming audio playback stopped")
                                    break
                                
                                # Send marker every 10 seconds for synchronization in recovery
                                if i % (10 * recovery_sr) < chunk_size and i > 0:
                                    current_time = time.perf_counter() - self.start_time
                                    if self.audio_marker_outlet is not None:
                                        try:
                                            marker_str = f"RECOVERY_LOOMING_SYNC,{self.participant_id},{current_time:.2f}"
                                            self.audio_marker_outlet.push_sample([marker_str])
                                        except Exception as e:
                                            print(f"Error sending sync marker: {e}")
                                
                                chunk = recovery_looming[i:min(i+chunk_size, len(recovery_looming))]
                                stream.write(chunk.astype(np.float32))
                        
                        # Send LSL marker for recovery looming end
                        if self.audio_marker_outlet is not None:
                            try:
                                current_time = time.perf_counter() - self.start_time
                                marker_str = f"RECOVERY_LOOMING_END,{self.participant_id},{current_time:.2f}"
                                self.audio_marker_outlet.push_sample([marker_str])
                                print(f"Sent LSL marker: {marker_str}")
                            except Exception as e:
                                print(f"Error sending LSL marker: {e}")
                        
                        print("Recovery looming audio playback completed")
                    
                except Exception as e:
                    print(f"Error playing looming audio: {e}")
                    traceback.print_exc()
            
            def play_tactile():
                try:
                    # Send LSL marker for tactile audio start
                    if self.audio_marker_outlet is not None:
                        try:
                            phase_name = "RECOVERY" if self.recovery_mode else "MAIN"
                            marker_str = f"TACTILE_START,{phase_name},{self.participant_id},0.0"
                            self.audio_marker_outlet.push_sample([marker_str])
                            print(f"Sent LSL marker: {marker_str}")
                        except Exception as e:
                            print(f"Error sending LSL marker: {e}")
                    
                    # Play main audio
                    with sd.OutputStream(samplerate=tactile_sr, channels=1, device=1) as stream:
                        chunk_size = 1024
                        for i in range(0, len(tactile_data), chunk_size):
                            if self.stop_audio:
                                print("Tactile audio playback stopped")
                                break
                            chunk = tactile_data[i:min(i+chunk_size, len(tactile_data))]
                            stream.write(chunk.astype(np.float32))
                    
                    # Send LSL marker for tactile audio end
                    if self.audio_marker_outlet is not None:
                        try:
                            phase_name = "RECOVERY" if self.recovery_mode else "MAIN"
                            current_time = time.perf_counter() - self.start_time
                            marker_str = f"TACTILE_END,{phase_name},{self.participant_id},{current_time:.2f}"
                            self.audio_marker_outlet.push_sample([marker_str])
                            print(f"Sent LSL marker: {marker_str}")
                        except Exception as e:
                            print(f"Error sending LSL marker: {e}")
                    
                    print("Tactile audio playback completed")
                    
                    # If recovery audio exists and we're not already in recovery mode, play that next
                    if self.recovery_audio_files and not self.recovery_mode and not self.stop_audio:
                        print("Playing recovery audio sequence for missed trials")
                        
                        # Send LSL marker for recovery tactile start
                        if self.audio_marker_outlet is not None:
                            try:
                                current_time = time.perf_counter() - self.start_time
                                marker_str = f"RECOVERY_TACTILE_START,{self.participant_id},{current_time:.2f}"
                                self.audio_marker_outlet.push_sample([marker_str])
                                print(f"Sent LSL marker: {marker_str}")
                            except Exception as e:
                                print(f"Error sending LSL marker: {e}")
                        
                        recovery_tactile, recovery_sr = sf.read(self.recovery_audio_files['tactile_path'])
                        with sd.OutputStream(samplerate=recovery_sr, channels=1, device=1) as stream:
                            chunk_size = 1024
                            for i in range(0, len(recovery_tactile), chunk_size):
                                if self.stop_audio:
                                    print("Recovery tactile audio playback stopped")
                                    break
                                chunk = recovery_tactile[i:min(i+chunk_size, len(recovery_tactile))]
                                stream.write(chunk.astype(np.float32))
                        
                        # Send LSL marker for recovery tactile end
                        if self.audio_marker_outlet is not None:
                            try:
                                current_time = time.perf_counter() - self.start_time
                                marker_str = f"RECOVERY_TACTILE_END,{self.participant_id},{current_time:.2f}"
                                self.audio_marker_outlet.push_sample([marker_str])
                                print(f"Sent LSL marker: {marker_str}")
                            except Exception as e:
                                print(f"Error sending LSL marker: {e}")
                        
                        print("Recovery tactile audio playback completed")
                    
                except Exception as e:
                    print(f"Error playing tactile audio: {e}")
                    traceback.print_exc()
            
            looming_thread = threading.Thread(target=play_looming, daemon=True)
            tactile_thread = threading.Thread(target=play_tactile, daemon=True)
            
            looming_thread.start()
            tactile_thread.start()
            
            # Calculate additional duration if recovery is generated
            additional_duration = 0
            if self.recovery_audio_files and not self.recovery_mode:
                recovery_looming, r_sr = sf.read(self.recovery_audio_files['looming_path'])
                additional_duration = len(recovery_looming) / r_sr
                print(f"Recovery audio adds {additional_duration:.2f} seconds to experiment")
                
            # Wait for audio to finish
            total_duration = max_duration + additional_duration
            end_time = self.start_time + total_duration + 1.0
            
            # Start a progress update thread
            progress_update_thread = threading.Thread(
                target=self.continuous_progress_update,
                args=(total_duration,),
                daemon=True
            )
            progress_update_thread.start()
            
            # Update status periodically
            while time.perf_counter() < end_time and not self.stop_audio and (looming_thread.is_alive() or tactile_thread.is_alive()):
                elapsed = time.perf_counter() - self.start_time
                progress_percent = min(100, (elapsed / total_duration) * 100)
                self.progress_var.set(progress_percent)
                
                minutes = int(elapsed // 60)
                seconds = int(elapsed % 60)
                total_minutes = int(total_duration // 60)
                total_seconds = int(total_duration % 60)
                
                # Show which phase we're in
                phase_text = ""
                if self.recovery_mode:
                    phase_text = "RECOVERY MODE - "
                elif elapsed > max_duration and self.recovery_audio_files:
                    phase_text = "RECOVERY TRIALS - "
                
                self.root.after(0, update_status, 
                               f"{phase_text}Experiment running - {minutes:02d}:{seconds:02d} / {total_minutes:02d}:{total_seconds:02d}")
                time.sleep(0.1)
            
            if not self.stop_audio:
                # Wait for threads to complete
                looming_thread.join(timeout=1.0)
                tactile_thread.join(timeout=1.0)
                
                # Send LSL marker for experiment end
                if self.audio_marker_outlet is not None:
                    try:
                        phase_name = "RECOVERY" if self.recovery_mode else "MAIN"
                        current_time = time.perf_counter() - self.start_time
                        marker_str = f"AUDIO_END,{phase_name},{self.participant_id},{current_time:.2f}"
                        self.audio_marker_outlet.push_sample([marker_str])
                        print(f"Sent LSL marker: {marker_str}")
                    except Exception as e:
                        print(f"Error sending LSL marker: {e}")
                
                print(f"Audio playback completed at {datetime.datetime.now().strftime('%H:%M:%S.%f')}")
                self.root.after(0, update_status, "Experiment completed - analyzing results...")
                
                # Enable finish button
                self.root.after(0, lambda: self.finish_button.config(state=tk.NORMAL, style="Green.TButton"))
                
                # Save the response data
                self.save_response_data()
                
                # Analyze results
                self.analyze_results()
            else:
                print("Audio playback was stopped by user")
                self.root.after(0, update_status, "Experiment stopped")
                
                # Send LSL marker for experiment stop
                if self.audio_marker_outlet is not None:
                    try:
                        phase_name = "RECOVERY" if self.recovery_mode else "MAIN"
                        current_time = time.perf_counter() - self.start_time
                        marker_str = f"AUDIO_STOP,{phase_name},{self.participant_id},{current_time:.2f}"
                        self.audio_marker_outlet.push_sample([marker_str])
                        print(f"Sent LSL marker: {marker_str}")
                    except Exception as e:
                        print(f"Error sending LSL marker: {e}")
            
        except Exception as e:
            print(f"ERROR during experiment: {e}")
            traceback.print_exc()
            
            def show_error(msg):
                self.status_var.set(msg)
            self.root.after(0, show_error, f"Error during experiment: {str(e)}")
            
        finally:
            self.experiment_running = False
            
            def enable_buttons():
                self.start_button.config(state=tk.NORMAL)
                # Only enable recovery checkbox if main experiment was just run (not recovery)
                if not self.recovery_mode:
                    self.recovery_checkbox.config(state=tk.NORMAL)
            self.root.after(0, enable_buttons)
            
            # Save the log regardless of completion
            self.save_raw_log()
    
    def continuous_progress_update(self, max_duration):
        """Continuously update the progress line during the experiment."""
        try:
            update_interval = 0.05  # Update every 50ms
            
            while self.experiment_running and not self.stop_audio:
                if self.start_time is not None:
                    elapsed = time.perf_counter() - self.start_time
                    if elapsed > max_duration:
                        break
                    # Schedule an update to the progress line
                    self.root.after(0, lambda t=elapsed: self.update_progress_line(t))
                    time.sleep(update_interval)
                else:
                    time.sleep(0.1)
            print("Progress update thread completed")
        except Exception as e:
            print(f"Error in progress update thread: {e}")
    
    def monitor_tactile_events(self):
        """Monitor and visualize tactile events during playback."""
        if not self.tactile_times:
            return
        try:
            # Create an array to track which events have been processed
            processed_events = [False] * len(self.tactile_times)
            
            while self.experiment_running and not self.stop_audio:
                current_time = time.perf_counter() - self.start_time
                
                for i, t_time in enumerate(self.tactile_times):
                    # Check if we're close to a tactile event time and haven't processed it yet
                    if not processed_events[i] and abs(current_time - t_time) < 0.1:
                        # Flash indicator
                        self.root.after(0, lambda: self.flash_indicator(self.tactile_indicator, "yellow"))
                        print(f"Tactile event detected at {current_time:.3f}s (expected {t_time:.3f}s)")
                        
                        # Send LSL marker
                        if self.tactile_marker_outlet is not None:
                            try:
                                # Get trial number if available
                                trial_num = "unknown"
                                if hasattr(self, 'tactile_times_data') and i < len(self.tactile_times_data):
                                    trial_num = str(self.tactile_times_data[i]['trial_number'])
                                
                                marker_str = f"TACTILE,{t_time:.6f},{self.participant_id},{trial_num}"
                                self.tactile_marker_outlet.push_sample([marker_str])
                                print(f"Sent LSL marker: {marker_str}")
                            except Exception as e:
                                print(f"Error sending LSL marker: {e}")
                        
                        # Mark as processed so we don't trigger it again
                        processed_events[i] = True
                        break
                
                # Check if we're near the end of the experiment and should prepare recovery
                if not self.recovery_mode and not self.recovery_generated:
                    audio_duration = max(self.tactile_times) + 30  # Estimate total duration
                    # If we're 10 seconds from the end and have unprocessed events
                    if audio_duration - current_time < 10:
                        # Prepare recovery session
                        self.prepare_recovery_session()
                        
                time.sleep(0.01)  # Faster update rate for more precise timing
        except Exception as e:
            print(f"Error in tactile monitor: {e}")
            traceback.print_exc()
    
    def analyze_results(self):
        """Analyze the results of the experiment."""
        print("\n===== ANALYZING EXPERIMENT RESULTS =====")
        print(f"Tactile stimuli count: {len(self.tactile_times)}")
        print(f"Mouse clicks count: {len(self.mouse_clicks)}")
        
        # Initialize counters
        correct_clicks = 0
        false_alarms = 0
        missed_stimuli = 0
        
        # Initialize result details
        correct_details = []
        false_alarm_details = []
        missed_details = []
        
        # Mark each stimulus as not responded yet
        stimulus_responded = [False] * len(self.tactile_times)
        
        # Check each mouse click
        for click_idx, click in enumerate(self.mouse_clicks):
            click_time = click["time"]
            
            # Initialize variables for finding the closest tactile stimulus
            closest_idx = -1
            closest_diff = float('inf')
            
            # Find the closest preceding tactile stimulus
            for i, stim_time in enumerate(self.tactile_times):
                time_diff = click_time - stim_time
                # Only consider tactile stimuli that preceded this click and are closest
                if 0 < time_diff < closest_diff:
                    closest_diff = time_diff
                    closest_idx = i
            
            # Determine if the click is a correct response or false alarm
            if closest_idx >= 0 and closest_diff <= RESPONSE_WINDOW:
                print(f"  ✓ CORRECT: Click at {click_time:.3f}s matches tactile at {self.tactile_times[closest_idx]:.3f}s (diff: {closest_diff:.3f}s)")
                correct_clicks += 1
                stimulus_responded[closest_idx] = True
                correct_details.append({
                    'click_time': click_time,
                    'tactile_time': self.tactile_times[closest_idx],
                    'difference': closest_diff
                })
            else:
                if closest_idx >= 0:
                    print(f"  ✗ FALSE ALARM: Click at {click_time:.3f}s is too late for tactile at {self.tactile_times[closest_idx]:.3f}s (diff: {closest_diff:.3f}s > {RESPONSE_WINDOW}s)")
                else:
                    print(f"  ✗ FALSE ALARM: Click at {click_time:.3f}s doesn't match any tactile stimulus")
                false_alarms += 1
                false_alarm_details.append({
                    'click_time': click_time,
                    'closest_tactile': self.tactile_times[closest_idx] if closest_idx >= 0 else None,
                    'difference': closest_diff if closest_idx >= 0 else None
                })
        
        # Check for missed stimuli (tactile events without click responses)
        missed_trials = []
        for i, responded in enumerate(stimulus_responded):
            if not responded:
                stim_time = self.tactile_times[i]
                print(f"✗ MISSED: Tactile at {stim_time:.3f}s had no response within {RESPONSE_WINDOW} seconds")
                missed_stimuli += 1
                missed_details.append({'tactile_time': stim_time})
                
                # Add to missed trials list for potential recovery
                missed_trials.append({
                    'trial_number': i + 1,
                    'tactile_time': stim_time
                })
        
        # Calculate hit rate
        total_tactile = len(self.tactile_times)
        hit_rate = (correct_clicks / total_tactile) if total_tactile > 0 else 0
        
        print(f"\nSummary: {correct_clicks}/{total_tactile} correct, {false_alarms} false alarms, {missed_stimuli} misses")
        
        # Create detailed diagnostic report
        diagnostic_report = f"===== DETAILED DIAGNOSTIC REPORT =====\n\n"
        diagnostic_report += f"PARTICIPANT ID: {self.participant_id}\n"
        diagnostic_report += f"PHASE: {'Recovery' if self.recovery_mode else 'Main'} Experiment\n"
        diagnostic_report += f"RESPONSE WINDOW: {RESPONSE_WINDOW} seconds\n\n"
        
        # Add correct responses section
        diagnostic_report += f"CORRECT RESPONSES ({correct_clicks}):\n"
        if correct_details:
            for i, detail in enumerate(correct_details):
                diagnostic_report += (
                    f"  {i+1}. Click at {detail['click_time']:.3f}s matched tactile "
                    f"at {detail['tactile_time']:.3f}s (diff: {detail['difference']:.3f}s)\n"
                )
        else:
            diagnostic_report += "  None\n"
        
        # Add false alarms section
        diagnostic_report += f"\nFALSE ALARMS ({false_alarms}):\n"
        if false_alarm_details:
            for i, detail in enumerate(false_alarm_details):
                if detail['closest_tactile'] is not None:
                    diagnostic_report += (
                        f"  {i+1}. Click at {detail['click_time']:.3f}s was too late for "
                        f"tactile at {detail['closest_tactile']:.3f}s (diff: {detail['difference']:.3f}s)\n"
                    )
                else:
                    diagnostic_report += (
                        f"  {i+1}. Click at {detail['click_time']:.3f}s didn't match any tactile stimulus\n"
                    )
        else:
            diagnostic_report += "  None\n"
        
        # Add missed stimuli section
        diagnostic_report += f"\nMISSED STIMULI ({missed_stimuli}):\n"
        if missed_details:
            for i, detail in enumerate(missed_details):
                diagnostic_report += (
                    f"  {i+1}. Tactile at {detail['tactile_time']:.3f}s had no response within {RESPONSE_WINDOW} seconds\n"
                )
        else:
            diagnostic_report += "  None\n"
        
        # Add conclusion
        diagnostic_report += "\nRESULTS SUMMARY:\n"
        diagnostic_report += f"  Hit rate: {hit_rate*100:.1f}% ({correct_clicks}/{total_tactile})\n"
        diagnostic_report += f"  False alarms: {false_alarms}\n"
        diagnostic_report += f"  Missed stimuli: {missed_stimuli}\n"
        
        if missed_stimuli > 0 and not self.recovery_mode:
            diagnostic_report += "\nRECOVERY RECOMMENDATION:\n"
            diagnostic_report += f"  {missed_stimuli} stimuli were missed. Consider running a recovery session.\n"
        
        self.diagnostic_report = diagnostic_report
        
        # Show results
        detailed_results = (
            f"Experiment Results:\n\n"
            f"Participant ID: {self.participant_id}\n"
            f"Phase: {'Recovery' if self.recovery_mode else 'Main'} Experiment\n\n"
            f"Correct responses: {correct_clicks} / {total_tactile} ({hit_rate*100:.1f}%)\n"
            f"False alarms: {false_alarms}\n"
            f"Missed stimuli: {missed_stimuli}\n\n"
        )
        
        # Add missed stimuli information
        if missed_stimuli > 0:
            if self.recovery_mode:
                detailed_results += "MISSED STIMULI DURING RECOVERY:\n"
            else:
                detailed_results += "MISSED STIMULI:\n"
                
            for i, detail in enumerate(missed_details[:5]):  # Show first 5 for brevity
                detailed_results += f"- Tactile at {detail['tactile_time']:.1f}s had no response\n"
            if len(missed_details) > 5:
                detailed_results += f"  (and {len(missed_details) - 5} more...)\n"
            detailed_results += "\n"
            
            if not self.recovery_mode:
                detailed_results += "RECOMMENDATION: Run a recovery session to catch missed trials.\n\n"
        
        # Show instructions
        detailed_results += "Click 'Show Detailed Diagnostics' for more information.\n"
        detailed_results += "Click 'Finish & Save Results' to save this data.\n"
        
        # Update UI
        self.results_var.set(detailed_results)
        self.results_frame.pack(fill=tk.X, padx=5, pady=5)
        
        # Show diagnostic button
        self.show_diagnostic_button.pack(side=tk.LEFT, padx=5)
        self.show_diagnostic_button.config(state=tk.NORMAL)
        
        # Save results with missed trials for potential recovery
        self.save_results(missed_trials)
    
    def save_raw_log(self):
        """Save the raw mouse click log to a CSV file."""
        if not self.mouse_clicks:
            return
        try:
            df = pd.DataFrame(self.mouse_clicks)
            df['participant_id'] = self.participant_id
            df['recovery_mode'] = self.recovery_mode
            df.to_csv(self.current_log_file, index=False)
            print(f"Saved raw mouse click log to {self.current_log_file}")
        except Exception as e:
            print(f"Error saving raw log: {e}")
    
    def save_response_data(self):
        """Save the participant's response data based on the original design CSV."""
        if not hasattr(self, 'original_design_df') or self.original_design_df is None:
            print("No original design data available for saving response data")
            return
        
        try:
            # Create a copy of the original design DataFrame
            result_df = self.original_design_df.copy()
            
            # Add reaction time and response columns
            result_df['responded'] = False
            result_df['reaction_time'] = None
            
            # Update with response data
            for trial_num, response_data in self.trial_responses.items():
                if response_data['responded']:
                    # Find the matching row
                    mask = result_df['trial_number'] == trial_num
                    if mask.any():
                        result_df.loc[mask, 'responded'] = True
                        result_df.loc[mask, 'reaction_time'] = response_data['reaction_time']
            
            # Save to CSV
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            phase = "recovery" if self.recovery_mode else "main"
            csv_filename = os.path.join(RESULTS_DIR, f"participant_{self.participant_id}_{phase}_response_data_{timestamp}.csv")
            
            result_df.to_csv(csv_filename, index=False)
            print(f"Saved response data to {csv_filename}")
            
        except Exception as e:
            print(f"Error saving response data: {e}")
            traceback.print_exc()
    
    def save_results(self, missed_trials):
        """
        Save experiment results including missed trials for potential recovery.
        
        Args:
            missed_trials: List of dictionaries with missed trial information
        """
        try:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            phase = "recovery" if self.recovery_mode else "main"
            
            # Store all data in a single dictionary
            results = {
                'participant_id': self.participant_id,
                'phase': phase,
                'timestamp': timestamp,
                'total_trials': len(self.tactile_times),
                'correct_responses': sum(1 for click in self.mouse_clicks if any(
                    0 < (click['time'] - t_time) <= RESPONSE_WINDOW for t_time in self.tactile_times
                )),
                'false_alarms': sum(1 for click in self.mouse_clicks if not any(
                    0 < (click['time'] - t_time) <= RESPONSE_WINDOW for t_time in self.tactile_times
                )),
                'missed_trials': missed_trials,
                'all_clicks': self.mouse_clicks,
                'tactile_times': self.tactile_times,
                'response_window': RESPONSE_WINDOW,
                'recovery_generated': self.recovery_generated,
                'recovery_files': self.recovery_audio_files
            }
            
            # Save to JSON
            results_file = os.path.join(RESULTS_DIR, f"participant_{self.participant_id}_results.json")
            with open(results_file, 'w') as f:
                json.dump(results, f, indent=2)
            
            print(f"Saved results to {results_file}")
            
        except Exception as e:
            print(f"Error saving results: {e}")
            traceback.print_exc()
    
    def prepare_recovery_session(self):
        """
        Analyze missed trials and prepare recovery audio on-the-fly.
        This is called during the experiment when nearing the end.
        """
        if self.recovery_mode or self.recovery_generated:
            return  # Already in recovery mode or already generated
        
        print("\n===== PREPARING RECOVERY SESSION =====")
        
        # Find missed trials
        missed_trials = []
        for trial_num, response_data in self.trial_responses.items():
            if response_data['trial_type'] != 'catch' and not response_data['responded']:
                missed_trials.append(response_data['trial_data'])
                print(f"  - Trial {trial_num} was missed")
        
        if not missed_trials:
            print("No missed trials found, recovery not needed")
            self.recovery_generated = True
            return
        
        # Generate recovery session audio
        try:
            print(f"Generating recovery audio for {len(missed_trials)} missed trials")
            
            # Create recovery directory for this participant
            participant_recovery_dir = os.path.join(RECOVERY_DIR, f"participant_{self.participant_id}")
            os.makedirs(participant_recovery_dir, exist_ok=True)
            
            # Save missed trials CSV
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            missed_csv_path = os.path.join(participant_recovery_dir, f"missed_trials_{timestamp}.csv")
            
            # Create a DataFrame from missed trials
            missed_df = pd.DataFrame(missed_trials)
            missed_df.to_csv(missed_csv_path, index=False)
            
            # Generate recovery audio files
            recovery_files = self.generate_recovery_audio(missed_trials, participant_recovery_dir)
            
            if recovery_files:
                print("Recovery audio generated successfully:")
                print(f"  - Looming: {recovery_files['looming_path']}")
                print(f"  - Tactile: {recovery_files['tactile_path']}")
                print(f"  - Tactile timestamps: {recovery_files['tactile_times']}")
                
                self.recovery_audio_files = recovery_files
                self.recovery_tactile_times = recovery_files['tactile_times']
                
                # Add recovery markers to timeline
                for t_time in self.recovery_tactile_times:
                    # Adjust time to account for main experiment duration
                    adjusted_time = self.tactile_times[-1] + 10 + t_time  # Add 10 second buffer
                    try:
                        self.add_timeline_marker(adjusted_time, "purple")
                    except Exception as e:
                        print(f"Error adding recovery tactile marker: {e}")
            
            self.recovery_generated = True
            
        except Exception as e:
            print(f"Error generating recovery audio: {e}")
            traceback.print_exc()
    
    def generate_recovery_audio(self, missed_trials, output_dir):
        """
        Generate recovery audio files for missed trials.
        
        Args:
            missed_trials: List of dictionaries with missed trial data
            output_dir: Directory to save the output files
            
        Returns:
            Dictionary with paths to generated files and tactile timestamps
        """
        if not missed_trials:
            return None
        
        try:
            # Ensure output directory exists
            os.makedirs(output_dir, exist_ok=True)
            
            # Base filenames
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            looming_path = os.path.join(output_dir, f"recovery_looming_{timestamp}.wav")
            tactile_path = os.path.join(output_dir, f"recovery_tactile_{timestamp}.wav")
            
            # Parameters
            interstimulus_interval = 6.0  # seconds between trials
            buffer_start = 5.0  # seconds of silence at the beginning
            buffer_end = 5.0  # seconds of silence at the end
            sample_rate = SAMPLE_RATE
            
            # Load tactile stimulus
            tactile_stimulus_path = os.path.join(STIMULUS_DIR, 'tactile_stimulus.wav')
            tactile_stimulus, tactile_sr = sf.read(tactile_stimulus_path)
            
            # Convert to mono if needed
            if len(tactile_stimulus.shape) > 1 and tactile_stimulus.shape[1] > 1:
                tactile_stimulus = np.mean(tactile_stimulus, axis=1)
            
            # Calculate total duration
            total_duration_sec = buffer_start + (len(missed_trials) * interstimulus_interval) + buffer_end
            total_samples = int(total_duration_sec * sample_rate)
            
            # Create empty arrays for audio
            if AUDIO_CHANNELS == 2:
                looming_audio = np.zeros((total_samples, 2), dtype=np.float32)
                tactile_audio = np.zeros((total_samples, 2), dtype=np.float32)
            else:
                looming_audio = np.zeros(total_samples, dtype=np.float32)
                tactile_audio = np.zeros(total_samples, dtype=np.float32)
            
            # List to track tactile event times
            tactile_times = []
            
            # Process each missed trial
            for i, trial in enumerate(missed_trials):
                # Calculate start time for this trial
                start_time_sec = buffer_start + (i * interstimulus_interval)
                start_sample = int(start_time_sec * sample_rate)
                
                # Determine looming stimulus type
                stim_type = trial.get('stimulus_type')
                if not stim_type or stim_type not in LOOMING_STIMULUS_FILES:
                    stim_type = 'white'  # default
                
                # Load looming stimulus
                looming_stim_path = os.path.join(STIMULUS_DIR, LOOMING_STIMULUS_FILES[stim_type])
                try:
                    looming_stim, looming_sr = sf.read(looming_stim_path)
                    
                    # Normalize if needed
                    max_val = np.max(np.abs(looming_stim))
                    if max_val > 0.95:
                        looming_stim = looming_stim * (0.95 / max_val)
                    
                    # Convert to mono if needed
                    if len(looming_stim.shape) > 1 and looming_stim.shape[1] > 1:
                        if AUDIO_CHANNELS == 1:
                            looming_stim = np.mean(looming_stim, axis=1)
                    
                    # Get end sample position
                    end_sample = start_sample + len(looming_stim)
                    
                    # Insert into looming audio (handle mono/stereo differences)
                    if len(looming_audio.shape) > 1 and len(looming_stim.shape) > 1:
                        # Both stereo
                        looming_audio[start_sample:end_sample] = looming_stim
                    elif len(looming_audio.shape) > 1 and len(looming_stim.shape) == 1:
                        # Output stereo, stimulus mono
                        for ch in range(looming_audio.shape[1]):
                            looming_audio[start_sample:end_sample, ch] = looming_stim
                    elif len(looming_audio.shape) == 1 and len(looming_stim.shape) > 1:
                        # Output mono, stimulus stereo
                        looming_audio[start_sample:end_sample] = looming_stim.mean(axis=1)
                    else:
                        # Both mono
                        looming_audio[start_sample:end_sample] = looming_stim
                    
                except Exception as e:
                    print(f"Error loading looming stimulus {stim_type}: {e}")
                
                # Determine tactile delay
                tactile_delay = trial.get('tactile_delay', 0.2)  # default 200ms
                if pd.isna(tactile_delay):
                    tactile_delay = 0.2
                
                # Calculate tactile stimulus position
                tactile_time_sec = start_time_sec + tactile_delay
                tactile_time_sample = int(tactile_time_sec * sample_rate)
                
                # Add tactile stimulus to audio
                tactile_end_sample = tactile_time_sample + len(tactile_stimulus)
                
                # Insert into tactile audio (handle mono/stereo differences)
                if len(tactile_audio.shape) > 1 and len(tactile_stimulus.shape) > 1:
                    # Both stereo
                    tactile_audio[tactile_time_sample:tactile_end_sample] = tactile_stimulus
                elif len(tactile_audio.shape) > 1 and len(tactile_stimulus.shape) == 1:
                    # Output stereo, stimulus mono
                    for ch in range(tactile_audio.shape[1]):
                        tactile_audio[tactile_time_sample:tactile_end_sample, ch] = tactile_stimulus
                elif len(tactile_audio.shape) == 1 and len(tactile_stimulus.shape) > 1:
                    # Output mono, stimulus stereo
                    tactile_audio[tactile_time_sample:tactile_end_sample] = tactile_stimulus.mean(axis=1)
                else:
                    # Both mono
                    tactile_audio[tactile_time_sample:tactile_end_sample] = tactile_stimulus
                
                # Record tactile event time
                tactile_times.append(tactile_time_sec)
            
            # Normalize audio
            max_looming = np.max(np.abs(looming_audio))
            if max_looming > 0.95:
                looming_audio = looming_audio * (0.95 / max_looming)
                print(f"Normalized looming audio (max: {max_looming:.4f})")
            
            max_tactile = np.max(np.abs(tactile_audio))
            if max_tactile > 0.95:
                tactile_audio = tactile_audio * (0.95 / max_tactile)
                print(f"Normalized tactile audio (max: {max_tactile:.4f})")
            
            # Save audio files
            sf.write(looming_path, looming_audio, sample_rate)
            sf.write(tactile_path, tactile_audio, sample_rate)
            
            print(f"Generated recovery audio files:")
            print(f"  - {looming_path} ({total_duration_sec:.2f}s)")
            print(f"  - {tactile_path} ({total_duration_sec:.2f}s)")
            print(f"  - {len(tactile_times)} tactile events")
            
            return {
                'looming_path': looming_path,
                'tactile_path': tactile_path,
                'tactile_times': tactile_times,
                'duration': total_duration_sec,
                'sample_rate': sample_rate
            }
            
        except Exception as e:
            print(f"Error generating recovery audio: {e}")
            traceback.print_exc()
            return None
    
    def finish_experiment(self):
        """Finish the experiment and save all results."""
        try:
            # Final save
            self.save_raw_log()
            self.save_response_data()
            
            # Show a completion message
            messagebox.showinfo(
                "Experiment Complete", 
                f"The experiment for participant {self.participant_id} has been completed and results saved."
            )
            
            # Reset UI
            self.results_frame.pack_forget()
            self.finish_button.pack_forget()
            self.show_diagnostic_button.pack_forget()
            self.progress_var.set(0)
            self.clear_timeline()
            
            # Update status
            self.status_var.set("Experiment completed. Select a participant to start again.")
            
            # Re-enable recovery checkbox if we just ran main experiment
            if not self.recovery_mode:
                self.check_recovery_data()
            
        except Exception as e:
            print(f"Error finishing experiment: {e}")
            messagebox.showerror("Error", f"There was an error finishing the experiment: {str(e)}")
            traceback.print_exc()

def main():
    app = PPSExperimentRunner()
    app.root.mainloop()

if __name__ == "__main__":
    main()

LSL streams initialized successfully
Found 100 available participants
Mouse click binding established

===== STARTING MAIN EXPERIMENT =====
Participant ID: 0
Recovery Mode: False
Clearing timeline markers...
Timeline cleared successfully
Using audio files:
- Looming: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_looming.wav
- Tactile: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_tactile.wav
Loading trial data from: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog\participant_0_design.csv
Loaded design with 206 trials
Loaded 180 tactile stimulus times
Loaded audio files:
- Looming: 1675.26s, 48000Hz, shape: (80412672, 2)
- Tactile: 1675.26s, 48000Hz, shape: (80412672, 2)
Converting looming audio from stereo to mono for playback
Converting tactile audio from stereo to mono for playback
Starting audio playback at

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_13400\2690749081.py", line 983, in play_looming
    with sd.OutputStream(samplerate=looming_sr, channels=1, device=0) as stream:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\sounddevice.py", line 1515, in __init__
    _StreamBase.__init__(self, kind='output', wrap_callback='array',
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\sounddevice.py", line 909, in __init__
    _check(_lib.Pa_OpenStream(self._ptr, iparameters, oparameters,
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\sounddevice.py", line 2796, in _check
    raise PortAudioError(errormsg, err)
sounddevice.PortAudioError: Error opening OutputStream: Invalid number of channels [PaErrorCode -9998]
Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_13400\2690749081.py", line 1080, in play_ta


===== STARTING RECOVERY EXPERIMENT =====
Participant ID: 0
Recovery Mode: True
Clearing timeline markers...
Timeline cleared successfully
Using audio files:
- Looming: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_looming.wav
- Tactile: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_tactile.wav
Using missed trials for recovery session
Loaded 0 tactile times for recovery
ERROR: No tactile stimulus times found

===== STARTING RECOVERY EXPERIMENT =====
Participant ID: 0
Recovery Mode: True
Clearing timeline markers...
Timeline cleared successfully
Using audio files:
- Looming: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_looming.wav
- Tactile: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_tactile.wav
Using missed trials 

KeyboardInterrupt: 

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Minimal Experiment Runner

This is a streamlined version focusing on core functionality:
- Loading and playing the two audio files
- Displaying the correct timeline length
- Showing the tactile stimulus times
- Capturing and displaying mouse clicks

Author: AI Assistant
"""

import os
import time
import tkinter as tk
from tkinter import ttk, messagebox
import sounddevice as sd
import soundfile as sf
import numpy as np
import pandas as pd
import threading
import datetime
import re
import glob
from pathlib import Path

# Configuration
BASE_DIR = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace"
EXPERIMENT_AUDIO_DIR = os.path.join(BASE_DIR, "Level1_AudioGeneration", "ExperimentAudio")
EXPERIMENT_LOG_DIR = os.path.join(BASE_DIR, "Level1_AudioGeneration", "ExperimentLog")
RESULTS_DIR = os.path.join(BASE_DIR, "Level2_RunExperiment", "Results")

# Ensure results directory exists
os.makedirs(RESULTS_DIR, exist_ok=True)

class MinimalExperimentRunner:
    def __init__(self):
        # Initialize variables
        self.participant_id = None
        self.available_participants = []
        self.experiment_running = False
        self.start_time = None
        self.mouse_clicks = []
        self.tactile_times = []
        self.timeline_markers = []
        self.click_count = 0
        self.audio_duration = 0  # Will be set based on loaded audio
        
        # Flag to control audio playback
        self.stop_audio = False
        
        # Scan for available participants
        self.scan_available_participants()
        
        # Create GUI
        self.create_gui()

    def scan_available_participants(self):
        """Scan for available participants based on design files."""
        self.available_participants = []
        
        # Look for participant design files
        pattern = os.path.join(EXPERIMENT_LOG_DIR, "participant_*_design.csv")
        
        for file_path in glob.glob(pattern):
            # Extract participant ID from filename
            match = re.search(r'participant_(\d+)_design\.csv', os.path.basename(file_path))
            if match:
                participant_id = int(match.group(1))
                
                # Check if corresponding audio files exist
                looming_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{participant_id}_design_looming.wav")
                tactile_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{participant_id}_design_tactile.wav")
                
                if os.path.exists(looming_file) and os.path.exists(tactile_file):
                    self.available_participants.append(participant_id)
        
        # Sort numerically
        self.available_participants.sort()
        
        print(f"Found {len(self.available_participants)} available participants")
    
    def create_gui(self):
        """Create the GUI."""
        self.root = tk.Tk()
        self.root.title("PPS Experiment Runner")
        
        # Set protocol for window close
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)
        
        # Get screen dimensions
        screen_width = self.root.winfo_screenwidth()
        screen_height = self.root.winfo_screenheight()
        
        # Set window size to 80% of screen
        window_width = int(screen_width * 0.8)
        window_height = int(screen_height * 0.8)
        self.root.geometry(f"{window_width}x{window_height}")
        
        # Main frame with padding
        main_frame = ttk.Frame(self.root, padding=10)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Title
        ttk.Label(main_frame, text="PPS Experiment Runner", font=("Arial", 16, "bold")).pack(pady=10)
        
        # Participant selection frame
        participant_frame = ttk.LabelFrame(main_frame, text="Participant Selection", padding=10)
        participant_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # Horizontal frame for participant selection
        selection_frame = ttk.Frame(participant_frame)
        selection_frame.pack(fill=tk.X, pady=5)
        
        # Participant ID selection
        ttk.Label(selection_frame, text="Participant ID:").pack(side=tk.LEFT, padx=5)
        
        self.participant_var = tk.StringVar()
        if self.available_participants:
            self.participant_var.set(str(self.available_participants[0]))
        
        participant_dropdown = ttk.Combobox(
            selection_frame, 
            textvariable=self.participant_var,
            values=[str(p) for p in self.available_participants],
            width=5
        )
        participant_dropdown.pack(side=tk.LEFT, padx=5)
        
        # Refresh button
        ttk.Button(
            selection_frame, 
            text="Refresh List",
            command=self.refresh_participants
        ).pack(side=tk.LEFT, padx=10)
        
        # Status display
        status_frame = ttk.Frame(main_frame)
        status_frame.pack(fill=tk.X, pady=5)
        
        self.status_var = tk.StringVar(value="Select a participant to begin")
        self.status_label = ttk.Label(status_frame, textvariable=self.status_var, 
                                     font=("Arial", 11))
        self.status_label.pack(pady=5)
        
        # Progress bar
        self.progress_var = tk.DoubleVar(value=0.0)
        self.progress_bar = ttk.Progressbar(
            main_frame, 
            orient="horizontal", 
            length=window_width-50,
            mode="determinate",
            variable=self.progress_var
        )
        self.progress_bar.pack(fill=tk.X, padx=10, pady=5)
        
        # Timeline frame
        timeline_frame = ttk.LabelFrame(main_frame, text="Experiment Timeline", padding=10)
        timeline_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # Canvas for timeline visualization
        self.timeline_canvas = tk.Canvas(timeline_frame, width=window_width-60, height=100, bg="white")
        self.timeline_canvas.pack(fill=tk.X, pady=5)
        
        # Create timeline
        timeline_start_x = 50
        self.timeline_end_x = window_width - 110
        self.timeline_y = 50
        self.timeline_canvas.create_line(timeline_start_x, self.timeline_y, self.timeline_end_x, self.timeline_y, width=2)
        
        # Add labels to the timeline
        self.timeline_canvas.create_text(timeline_start_x - 40, self.timeline_y, text="Timeline:", font=("Arial", 9, "bold"))
        
        # Create progress indicator (vertical line)
        self.progress_line = self.timeline_canvas.create_line(
            timeline_start_x, self.timeline_y - 20, timeline_start_x, self.timeline_y + 20, 
            width=3, fill="green", state="hidden"
        )
        
        # Mouse click visualization area
        click_frame = ttk.LabelFrame(main_frame, text="Mouse Click Area", padding=10)
        click_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Canvas for visualizing mouse clicks
        self.click_canvas = tk.Canvas(click_frame, bg="lightyellow")
        self.click_canvas.pack(fill=tk.BOTH, expand=True, pady=5)
        
        # Add text to the click area
        self.click_canvas.create_text(
            window_width//2 - 30, 50,
            text="CLICK HERE WHEN YOU HEAR THE TACTILE STIMULUS",
            font=("Arial", 14, "bold"), fill="blue"
        )
        
        # Add click counter
        self.click_counter_text = self.click_canvas.create_text(
            100, 20,
            text="Clicks: 0",
            font=("Arial", 12), fill="black"
        )
        
        # Bind mouse clicks
        self.click_canvas.bind("<Button-1>", self.on_mouse_click)
        
        # Control buttons frame
        control_frame = ttk.Frame(main_frame)
        control_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # Start button
        self.start_button = ttk.Button(
            control_frame, text="START EXPERIMENT",
            command=self.start_experiment, width=20,
            style="TButton"
        )
        self.start_button.pack(side=tk.LEFT, padx=10)
        
        # Stop button
        self.stop_button = ttk.Button(
            control_frame, text="STOP",
            command=self.stop_experiment, width=10,
            state=tk.DISABLED
        )
        self.stop_button.pack(side=tk.LEFT, padx=10)
        
        # Quit button
        ttk.Button(
            control_frame, text="QUIT",
            command=self.on_close, width=10
        ).pack(side=tk.RIGHT, padx=10)
    
    def refresh_participants(self):
        """Refresh the list of available participants."""
        previous_selection = self.participant_var.get()
        
        # Scan for available participants
        self.scan_available_participants()
        
        # Update dropdown values
        participant_dropdown = self.root.nametowidget(
            self.participant_var.winfo_pathname(self.participant_var.winfo_id())
        )
        participant_dropdown['values'] = [str(p) for p in self.available_participants]
        
        # Try to keep previous selection if it still exists
        if previous_selection in [str(p) for p in self.available_participants]:
            self.participant_var.set(previous_selection)
        elif self.available_participants:
            self.participant_var.set(str(self.available_participants[0]))
        else:
            self.participant_var.set("")
    
    def on_close(self):
        """Handle window close event."""
        if self.experiment_running:
            if messagebox.askyesno("Quit", "Experiment is running. Are you sure you want to quit?"):
                # Stop audio playback when closing the window
                self.stop_audio = True
                sd.stop()
                print("Audio playback stopped")
                self.root.destroy()
        else:
            self.root.destroy()
    
    def on_mouse_click(self, event):
        """Handle mouse click events during the experiment."""
        if not self.experiment_running or self.start_time is None:
            return
            
        # Calculate time since experiment start
        current_time = time.perf_counter() - self.start_time
        
        # Add to mouse clicks list
        self.mouse_clicks.append({
            "time": current_time,
            "timestamp": datetime.datetime.now().isoformat(),
            "x": event.x,
            "y": event.y
        })
        
        # Update click count
        self.click_count += 1
        self.click_canvas.itemconfig(
            self.click_counter_text, 
            text=f"Clicks: {self.click_count}"
        )
        
        # Create a visual indication of the click
        click_x, click_y = event.x, event.y
        circle = self.click_canvas.create_oval(
            click_x-10, click_y-10, click_x+10, click_y+10, 
            fill="red", outline="black"
        )
        
        # Fade out the circle after a short time
        self.root.after(500, lambda c=circle: self.click_canvas.delete(c))
        
        # Add click marker to timeline
        self.add_timeline_marker(current_time, "red")
        
        print(f"Mouse click at {current_time:.3f} seconds")
    
    def add_timeline_marker(self, time_sec, color):
        """Add a marker to the timeline at the specified time."""
        if self.audio_duration <= 0:
            return
            
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        # Calculate position based on time
        x_pos = timeline_start_x + min(time_sec / self.audio_duration, 1.0) * timeline_width
        
        # Create marker
        marker = self.timeline_canvas.create_oval(
            x_pos-4, self.timeline_y-4, x_pos+4, self.timeline_y+4, 
            fill=color, outline="black", width=1
        )
        self.timeline_markers.append(marker)
        
        # Update progress line
        self.timeline_canvas.coords(
            self.progress_line, 
            x_pos, self.timeline_y-20, 
            x_pos, self.timeline_y+20
        )
        self.timeline_canvas.itemconfig(self.progress_line, state="normal")
    
    def parse_timestamp(self, timestamp_str):
        """Parse timestamp in format MM:SS.S to seconds."""
        if pd.isna(timestamp_str):
            return None
            
        match = re.match(r'(\d+):(\d+\.\d+)', timestamp_str)
        if match:
            minutes, seconds = match.groups()
            return float(minutes) * 60 + float(seconds)
        return None
    
    def load_tactile_times(self, participant_id):
        """Load tactile stimulus times from the design file."""
        try:
            # Load from design file
            design_file = os.path.join(EXPERIMENT_LOG_DIR, f"participant_{participant_id}_design.csv")
            print(f"Loading trial data from: {design_file}")
            
            design_df = pd.read_csv(design_file)
            print(f"Loaded design with {len(design_df)} rows")
            
            # Filter for trials with tactile stimuli (exclude catch trials)
            tactile_trials = design_df[design_df['trial_type'] != 'catch']
            
            # Extract tactile stimulus times
            tactile_times = []
            for _, row in tactile_trials.iterrows():
                if 'tactile_stimulus_timestamp' in row and pd.notna(row['tactile_stimulus_timestamp']):
                    ts_str = row['tactile_stimulus_timestamp']
                    time_sec = self.parse_timestamp(ts_str)
                    if time_sec is not None:
                        tactile_times.append(time_sec)
            
            print(f"Loaded {len(tactile_times)} tactile stimulus times")
            return tactile_times
        except Exception as e:
            print(f"Error loading tactile times: {e}")
            return []
    
    def update_timeline_with_duration(self):
        """Update the timeline with markers based on audio duration."""
        if self.audio_duration <= 0:
            return
            
        # Clear existing time markers
        for item in self.timeline_canvas.find_all():
            if self.timeline_canvas.type(item) == "text" and item != self.click_counter_text:
                self.timeline_canvas.delete(item)
        
        # Add time markers every minute
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        # Determine a suitable interval based on duration
        if self.audio_duration < 60:  # Less than 1 minute
            interval = 10  # 10 second intervals
        elif self.audio_duration < 300:  # Less than 5 minutes
            interval = 30  # 30 second intervals
        else:
            interval = 60  # 1 minute intervals
        
        for sec in range(0, int(self.audio_duration) + interval, interval):
            if sec > self.audio_duration:
                break
                
            x_pos = timeline_start_x + (sec / self.audio_duration) * timeline_width
            
            # Create tick mark
            self.timeline_canvas.create_line(x_pos, self.timeline_y - 5, x_pos, self.timeline_y + 5, width=1)
            
            # Add time label
            if interval >= 60:
                # Show as minutes for longer intervals
                self.timeline_canvas.create_text(x_pos, self.timeline_y + 15, 
                                               text=f"{sec//60}m", 
                                               font=("Arial", 8))
            else:
                # Show as seconds for shorter intervals
                self.timeline_canvas.create_text(x_pos, self.timeline_y + 15, 
                                               text=f"{sec}s", 
                                               font=("Arial", 8))
        
        # Add tactile event markers
        for t_time in self.tactile_times:
            self.add_timeline_marker(t_time, "blue")
    
    def clear_timeline(self):
        """Clear all markers from the timeline."""
        for marker in self.timeline_markers:
            self.timeline_canvas.delete(marker)
        self.timeline_markers = []
        
        # Hide progress line
        self.timeline_canvas.itemconfig(self.progress_line, state="hidden")
    
    def update_progress(self, elapsed_time):
        """Update progress bar and timeline progress line."""
        if self.audio_duration <= 0:
            return
            
        # Update progress bar
        progress_percent = min(100, (elapsed_time / self.audio_duration) * 100)
        self.progress_var.set(progress_percent)
        
        # Update progress line on timeline
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        x_pos = timeline_start_x + min(elapsed_time / self.audio_duration, 1.0) * timeline_width
        
        self.timeline_canvas.coords(
            self.progress_line, 
            x_pos, self.timeline_y - 20, 
            x_pos, self.timeline_y + 20
        )
        self.timeline_canvas.itemconfig(self.progress_line, state="normal")
    
    def start_experiment(self):
        """Start the experiment."""
        # Get participant ID
        try:
            self.participant_id = int(self.participant_var.get())
        except (ValueError, TypeError):
            messagebox.showerror("Error", "Please select a valid participant ID")
            return
        
        print(f"\n===== STARTING EXPERIMENT FOR PARTICIPANT {self.participant_id} =====")
        
        # Reset state
        self.stop_audio = False
        self.mouse_clicks = []
        self.click_count = 0
        self.experiment_running = True
        self.clear_timeline()
        
        # Update status
        self.status_var.set(f"Starting experiment for participant {self.participant_id}...")
        
        # Update UI
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        self.click_canvas.itemconfig(self.click_counter_text, text=f"Clicks: 0")
        
        # Run experiment in background thread
        threading.Thread(target=self.run_experiment, daemon=True).start()
    
    def stop_experiment(self):
        """Stop the experiment."""
        if not self.experiment_running:
            return
            
        self.stop_audio = True
        sd.stop()
        print("Experiment stopped manually")
        
        # Update status
        self.status_var.set("Experiment stopped")
        
        # Update UI
        self.start_button.config(state=tk.NORMAL)
        self.stop_button.config(state=tk.DISABLED)
        
        # Save click data
        self.save_click_data()
    
    def run_experiment(self):
        """Run the experiment (play audio files and track responses)."""
        try:
            # Get audio file paths
            looming_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{self.participant_id}_design_looming.wav")
            tactile_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{self.participant_id}_design_tactile.wav")
            
            print(f"Using audio files:")
            print(f"- Looming: {looming_file}")
            print(f"- Tactile: {tactile_file}")
            
            # Verify files exist
            if not os.path.exists(looming_file) or not os.path.exists(tactile_file):
                self.status_var.set(f"Error: Audio files not found")
                print(f"ERROR: Missing audio files")
                self.experiment_running = False
                self.root.after(0, lambda: self.start_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                return
            
            # Load tactile stimulus times
            self.tactile_times = self.load_tactile_times(self.participant_id)
            if not self.tactile_times:
                self.status_var.set(f"Warning: No tactile stimulus times found")
                print(f"WARNING: No tactile stimulus times found")
            
            # Load audio files to get duration
            looming_data, looming_sr = sf.read(looming_file)
            tactile_data, tactile_sr = sf.read(tactile_file)
            
            # Set audio duration for timeline
            self.audio_duration = len(looming_data) / looming_sr
            print(f"Audio duration: {self.audio_duration:.2f} seconds")
            
            # Update timeline based on duration
            self.root.after(0, self.update_timeline_with_duration)
            
            # Convert stereo to mono if needed
            if len(looming_data.shape) > 1 and looming_data.shape[1] > 1:
                looming_data = np.mean(looming_data, axis=1)
            if len(tactile_data.shape) > 1 and tactile_data.shape[1] > 1:
                tactile_data = np.mean(tactile_data, axis=1)
            
            # Set start time just before playback
            self.start_time = time.perf_counter()
            print(f"Starting audio playback at {datetime.datetime.now().strftime('%H:%M:%S.%f')}")
            
            # Play both audio files in separate threads
            def play_looming():
                try:
                    with sd.OutputStream(samplerate=looming_sr, channels=1, device=0) as stream:
                        chunk_size = 1024
                        for i in range(0, len(looming_data), chunk_size):
                            if self.stop_audio:
                                print("Looming audio playback stopped")
                                break
                            chunk = looming_data[i:min(i+chunk_size, len(looming_data))]
                            stream.write(chunk.astype(np.float32))
                    print("Looming audio playback completed")
                except Exception as e:
                    print(f"Error playing looming audio: {e}")
            
            def play_tactile():
                try:
                    with sd.OutputStream(samplerate=tactile_sr, channels=1, device=1) as stream:
                        chunk_size = 1024
                        for i in range(0, len(tactile_data), chunk_size):
                            if self.stop_audio:
                                print("Tactile audio playback stopped")
                                break
                            chunk = tactile_data[i:min(i+chunk_size, len(tactile_data))]
                            stream.write(chunk.astype(np.float32))
                    print("Tactile audio playback completed")
                except Exception as e:
                    print(f"Error playing tactile audio: {e}")
            
            looming_thread = threading.Thread(target=play_looming, daemon=True)
            tactile_thread = threading.Thread(target=play_tactile, daemon=True)
            
            looming_thread.start()
            tactile_thread.start()
            
            # Progress update loop
            self.status_var.set("Experiment running - click when you hear a tactile stimulus")
            
            # Update progress bar and timeline periodically
            update_interval = 0.1  # seconds
            end_time = self.start_time + self.audio_duration + 2.0  # Add a small buffer
            
            while (time.perf_counter() < end_time and 
                   not self.stop_audio and 
                   (looming_thread.is_alive() or tactile_thread.is_alive())):
                
                elapsed = time.perf_counter() - self.start_time
                
                # Update progress UI
                self.root.after(0, lambda t=elapsed: self.update_progress(t))
                
                # Update status with time
                minutes = int(elapsed // 60)
                seconds = int(elapsed % 60)
                total_minutes = int(self.audio_duration // 60)
                total_seconds = int(self.audio_duration % 60)
                
                self.root.after(0, lambda m=minutes, s=seconds, tm=total_minutes, ts=total_seconds: 
                              self.status_var.set(f"Experiment running - {m:02d}:{s:02d} / {tm:02d}:{ts:02d}"))
                
                time.sleep(update_interval)
            
            # Wait for threads to complete
            looming_thread.join(timeout=1.0)
            tactile_thread.join(timeout=1.0)
            
            if not self.stop_audio:
                print(f"Audio playback completed at {datetime.datetime.now().strftime('%H:%M:%S.%f')}")
                self.status_var.set("Experiment completed")
                
                # Save click data
                self.save_click_data()
            
        except Exception as e:
            print(f"ERROR during experiment: {e}")
            import traceback
            traceback.print_exc()
            self.status_var.set(f"Error: {str(e)}")
        
        finally:
            self.experiment_running = False
            
            # Update UI
            self.root.after(0, lambda: self.start_button.config(state=tk.NORMAL))
            self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
    
    def save_click_data(self):
        """Save the mouse click data to CSV."""
        if not self.mouse_clicks:
            print("No click data to save")
            return
            
        try:
            # Create DataFrame from mouse clicks
            df = pd.DataFrame(self.mouse_clicks)
            
            # Add participant ID
            df['participant_id'] = self.participant_id
            
            # Save to CSV
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = os.path.join(RESULTS_DIR, f"participant_{self.participant_id}_clicks_{timestamp}.csv")
            
            df.to_csv(filename, index=False)
            print(f"Saved click data to {filename}")
            
            # Show success message
            self.status_var.set(f"Click data saved to {os.path.basename(filename)}")
            
        except Exception as e:
            print(f"Error saving click data: {e}")
            self.status_var.set(f"Error saving click data: {str(e)}")

def main():
    app = MinimalExperimentRunner()
    app.root.mainloop()

if __name__ == "__main__":
    main()

Found 100 available participants

===== STARTING EXPERIMENT FOR PARTICIPANT 0 =====
Using audio files:
- Looming: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_looming.wav
- Tactile: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_tactile.wav
Loading trial data from: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog\participant_0_design.csv
Loaded design with 206 rows
Loaded 180 tactile stimulus times
Audio duration: 1675.26 seconds
Starting audio playback at 22:37:01.624775
Mouse click at 0.000 seconds
Error playing looming audio: Error opening OutputStream: Invalid number of channels [PaErrorCode -9998]
Error playing tactile audio: Error opening OutputStream: Invalid number of channels [PaErrorCode -9998]
ERROR during experiment: main thread is not in main loop


Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_13400\960481509.py", line 577, in run_experiment
    self.status_var.set("Experiment running - click when you hear a tactile stimulus")
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\tkinter\__init__.py", line 424, in set
    return self._tk.globalsetvar(self._name, value)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: main thread is not in main loop
Exception in thread Thread-33 (run_experiment):
Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_13400\960481509.py", line 577, in run_experiment
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\tkinter\__init__.py", line 424, in set
    return self._tk.globalsetvar(self._name, value)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: main thread is not in main loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "


===== STARTING EXPERIMENT FOR PARTICIPANT 0 =====
Using audio files:
- Looming: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_looming.wav
- Tactile: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_tactile.wav
Loading trial data from: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog\participant_0_design.csv
Loaded design with 206 rows
Loaded 180 tactile stimulus times
Audio duration: 1675.26 seconds
Starting audio playback at 22:37:09.807374
Error playing looming audio: Error opening OutputStream: Invalid number of channels [PaErrorCode -9998]
Error playing tactile audio: Error opening OutputStream: Invalid number of channels [PaErrorCode -9998]
ERROR during experiment: main thread is not in main loop


Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_13400\960481509.py", line 577, in run_experiment
    self.status_var.set("Experiment running - click when you hear a tactile stimulus")
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\tkinter\__init__.py", line 424, in set
    return self._tk.globalsetvar(self._name, value)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: main thread is not in main loop
Exception in thread Thread-36 (run_experiment):
Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_13400\960481509.py", line 577, in run_experiment
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\tkinter\__init__.py", line 424, in set
    return self._tk.globalsetvar(self._name, value)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: main thread is not in main loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "

In [8]:
def parse_timestamp(self, timestamp_str):
        """Parse timestamp in format MM:SS.S to seconds."""
        if pd.isna(timestamp_str):
            return None
            
        match = re.match(r'(\d+):(\d+\.\d+)', timestamp_str)
        if match:
            minutes, seconds = match.groups()
            return float(minutes) * 60 + float(seconds)
        return None
    
    def load_tactile_times(self, participant_id):
        """Load tactile stimulus times from the design file."""
        try:
            # Load from design file
            design_file = os.path.join(EXPERIMENT_LOG_DIR, f"participant_{participant_id}_design.csv")
            print(f"Loading trial data from: {design_file}")
            
            design_df = pd.read_csv(design_file)
            print(f"Loaded design with {len(design_df)} rows")
            
            # Filter for trials with tactile stimuli (exclude catch trials)
            tactile_trials = design_df[design_df['trial_type'] != 'catch']
            
            # Extract tactile stimulus times
            tactile_times = []
            for _, row in tactile_trials.iterrows():
                if 'tactile_stimulus_timestamp' in row and pd.notna(row['tactile_stimulus_timestamp']):
                    ts_str = row['tactile_stimulus_timestamp']
                    time_sec = self.parse_timestamp(ts_str)
                    if time_sec is not None:
                        tactile_times.append(time_sec)
            
            print(f"Loaded {len(tactile_times)} tactile stimulus times")
            return tactile_times
        except Exception as e:
            print(f"Error loading tactile times: {e}")
            traceback.print_exc()
            return []#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Minimal Experiment Runner

This is a streamlined version focusing on core functionality:
- Loading and playing the two audio files
- Displaying the correct timeline length
- Showing the tactile stimulus times
- Capturing and displaying mouse clicks

Author: AI Assistant
"""

import os
import time
import tkinter as tk
from tkinter import ttk, messagebox
import sounddevice as sd
import soundfile as sf
import numpy as np
import pandas as pd
import threading
import datetime
import re
import glob
from pathlib import Path

# Configuration
BASE_DIR = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace"
EXPERIMENT_AUDIO_DIR = os.path.join(BASE_DIR, "Level1_AudioGeneration", "ExperimentAudio")
EXPERIMENT_LOG_DIR = os.path.join(BASE_DIR, "Level1_AudioGeneration", "ExperimentLog")
RESULTS_DIR = os.path.join(BASE_DIR, "Level2_RunExperiment", "Results")

# Ensure results directory exists
os.makedirs(RESULTS_DIR, exist_ok=True)

class MinimalExperimentRunner:
    def __init__(self):
        # Initialize variables
        self.participant_id = None
        self.available_participants = []
        self.experiment_running = False
        self.start_time = None
        self.mouse_clicks = []
        self.tactile_times = []
        self.timeline_markers = []
        self.click_count = 0
        self.audio_duration = 0  # Will be set based on loaded audio
        
        # Flag to control audio playback
        self.stop_audio = False
        
        # Scan for available participants
        self.scan_available_participants()
        
        # Create GUI
        self.create_gui()

    def scan_available_participants(self):
        """Scan for available participants based on design files."""
        self.available_participants = []
        
        # Look for participant design files
        pattern = os.path.join(EXPERIMENT_LOG_DIR, "participant_*_design.csv")
        
        for file_path in glob.glob(pattern):
            # Extract participant ID from filename
            match = re.search(r'participant_(\d+)_design\.csv', os.path.basename(file_path))
            if match:
                participant_id = int(match.group(1))
                
                # Check if corresponding audio files exist
                looming_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{participant_id}_design_looming.wav")
                tactile_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{participant_id}_design_tactile.wav")
                
                if os.path.exists(looming_file) and os.path.exists(tactile_file):
                    self.available_participants.append(participant_id)
        
        # Sort numerically
        self.available_participants.sort()
        
        print(f"Found {len(self.available_participants)} available participants")
    
    def create_gui(self):
        """Create the GUI."""
        self.root = tk.Tk()
        self.root.title("PPS Experiment Runner")
        
        # Set protocol for window close
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)
        
        # Get screen dimensions
        screen_width = self.root.winfo_screenwidth()
        screen_height = self.root.winfo_screenheight()
        
        # Set window size to 80% of screen
        window_width = int(screen_width * 0.8)
        window_height = int(screen_height * 0.8)
        self.root.geometry(f"{window_width}x{window_height}")
        
        # Main frame with padding
        main_frame = ttk.Frame(self.root, padding=10)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Title
        ttk.Label(main_frame, text="PPS Experiment Runner", font=("Arial", 16, "bold")).pack(pady=10)
        
        # Control buttons frame at the TOP
        control_frame = ttk.LabelFrame(main_frame, text="Experiment Control", padding=10)
        control_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # Start button
        self.start_button = ttk.Button(
            control_frame, text="START EXPERIMENT",
            command=self.start_experiment, width=20
        )
        self.start_button.pack(side=tk.LEFT, padx=10, pady=5)
        
        # Stop button
        self.stop_button = ttk.Button(
            control_frame, text="STOP",
            command=self.stop_experiment, width=10,
            state=tk.DISABLED
        )
        self.stop_button.pack(side=tk.LEFT, padx=10, pady=5)
        
        # Quit button
        ttk.Button(
            control_frame, text="QUIT",
            command=self.on_close, width=10
        ).pack(side=tk.RIGHT, padx=10, pady=5)
        
        # Participant selection frame
        participant_frame = ttk.LabelFrame(main_frame, text="Participant Selection", padding=10)
        participant_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # Horizontal frame for participant selection
        selection_frame = ttk.Frame(participant_frame)
        selection_frame.pack(fill=tk.X, pady=5)
        
        # Participant ID selection
        ttk.Label(selection_frame, text="Participant ID:").pack(side=tk.LEFT, padx=5)
        
        self.participant_var = tk.StringVar()
        if self.available_participants:
            self.participant_var.set(str(self.available_participants[0]))
        
        participant_dropdown = ttk.Combobox(
            selection_frame, 
            textvariable=self.participant_var,
            values=[str(p) for p in self.available_participants],
            width=5
        )
        participant_dropdown.pack(side=tk.LEFT, padx=5)
        
        # Refresh button
        ttk.Button(
            selection_frame, 
            text="Refresh List",
            command=self.refresh_participants
        ).pack(side=tk.LEFT, padx=10)
        
        # Status display
        status_frame = ttk.Frame(main_frame)
        status_frame.pack(fill=tk.X, pady=5)
        
        self.status_var = tk.StringVar(value="Select a participant and press START EXPERIMENT")
        self.status_label = ttk.Label(status_frame, textvariable=self.status_var, 
                                     font=("Arial", 11, "bold"))
        self.status_label.pack(pady=5)
        
        # Progress bar
        self.progress_var = tk.DoubleVar(value=0.0)
        self.progress_bar = ttk.Progressbar(
            main_frame, 
            orient="horizontal", 
            length=window_width-50,
            mode="determinate",
            variable=self.progress_var
        )
        self.progress_bar.pack(fill=tk.X, padx=10, pady=5)
        
        # Split Timeline frame - two timelines for better visualization
        timeline_frame = ttk.LabelFrame(main_frame, text="Experiment Timeline", padding=10)
        timeline_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # First half timeline
        timeline_label1 = ttk.Label(timeline_frame, text="First Half:")
        timeline_label1.pack(anchor=tk.W, pady=(0, 2))
        
        self.timeline_canvas1 = tk.Canvas(timeline_frame, width=window_width-60, height=60, bg="white")
        self.timeline_canvas1.pack(fill=tk.X, pady=2)
        
        # Second half timeline
        timeline_label2 = ttk.Label(timeline_frame, text="Second Half:")
        timeline_label2.pack(anchor=tk.W, pady=(5, 2))
        
        self.timeline_canvas2 = tk.Canvas(timeline_frame, width=window_width-60, height=60, bg="white")
        self.timeline_canvas2.pack(fill=tk.X, pady=2)
        
        # Create timelines
        timeline_start_x = 50
        self.timeline_end_x = window_width - 110
        
        # First timeline
        self.timeline_y1 = 30
        self.timeline_canvas1.create_line(timeline_start_x, self.timeline_y1, self.timeline_end_x, self.timeline_y1, width=2)
        self.timeline_canvas1.create_text(timeline_start_x - 40, self.timeline_y1, text="0:00", font=("Arial", 8))
        
        # First timeline progress indicator
        self.progress_line1 = self.timeline_canvas1.create_line(
            timeline_start_x, self.timeline_y1 - 15, timeline_start_x, self.timeline_y1 + 15, 
            width=3, fill="green", state="hidden"
        )
        
        # Second timeline
        self.timeline_y2 = 30
        self.timeline_canvas2.create_line(timeline_start_x, self.timeline_y2, self.timeline_end_x, self.timeline_y2, width=2)
        
        # Second timeline progress indicator
        self.progress_line2 = self.timeline_canvas2.create_line(
            timeline_start_x, self.timeline_y2 - 15, timeline_start_x, self.timeline_y2 + 15, 
            width=3, fill="green", state="hidden"
        )
        
        # Initialize markers list for each timeline
        self.timeline_markers1 = []
        self.timeline_markers2 = []
        
        # Mouse click visualization area
        click_frame = ttk.LabelFrame(main_frame, text="Mouse Click Area", padding=10)
        click_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Canvas for visualizing mouse clicks
        self.click_canvas = tk.Canvas(click_frame, bg="lightyellow")
        self.click_canvas.pack(fill=tk.BOTH, expand=True, pady=5)
        
        # Add text to the click area
        self.click_canvas.create_text(
            window_width//2 - 30, 50,
            text="CLICK HERE WHEN YOU HEAR THE TACTILE STIMULUS",
            font=("Arial", 14, "bold"), fill="blue"
        )
        
        # Add click counter
        self.click_counter_text = self.click_canvas.create_text(
            100, 20,
            text="Clicks: 0",
            font=("Arial", 12), fill="black"
        )
        
        # Bind mouse clicks
        self.click_canvas.bind("<Button-1>", self.on_mouse_click)
    
    def refresh_participants(self):
        """Refresh the list of available participants."""
        previous_selection = self.participant_var.get()
        
        # Scan for available participants
        self.scan_available_participants()
        
        # Update dropdown values
        participant_dropdown = self.root.nametowidget(
            self.participant_var.winfo_pathname(self.participant_var.winfo_id())
        )
        participant_dropdown['values'] = [str(p) for p in self.available_participants]
        
        # Try to keep previous selection if it still exists
        if previous_selection in [str(p) for p in self.available_participants]:
            self.participant_var.set(previous_selection)
        elif self.available_participants:
            self.participant_var.set(str(self.available_participants[0]))
        else:
            self.participant_var.set("")
    
    def on_close(self):
        """Handle window close event."""
        if self.experiment_running:
            if messagebox.askyesno("Quit", "Experiment is running. Are you sure you want to quit?"):
                # Stop audio playback when closing the window
                self.stop_audio = True
                sd.stop()
                print("Audio playback stopped")
                self.root.destroy()
        else:
            self.root.destroy()
    
    def on_mouse_click(self, event):
        """Handle mouse click events during the experiment."""
        if not self.experiment_running or self.start_time is None:
            return
            
        # Calculate time since experiment start
        current_time = time.perf_counter() - self.start_time
        
        # Add to mouse clicks list
        self.mouse_clicks.append({
            "time": current_time,
            "timestamp": datetime.datetime.now().isoformat(),
            "x": event.x,
            "y": event.y
        })
        
        # Update click count
        self.click_count += 1
        self.click_canvas.itemconfig(
            self.click_counter_text, 
            text=f"Clicks: {self.click_count}"
        )
        
        # Create a visual indication of the click
        click_x, click_y = event.x, event.y
        circle = self.click_canvas.create_oval(
            click_x-10, click_y-10, click_x+10, click_y+10, 
            fill="red", outline="black"
        )
        
        # Fade out the circle after a short time
        self.root.after(500, lambda c=circle: self.click_canvas.delete(c))
        
        # Add click marker to timeline
        self.add_timeline_marker(current_time, "red")
        
        print(f"Mouse click at {current_time:.3f} seconds")
    
    def add_timeline_marker(self, time_sec, color):
        """Add a marker to the timeline at the specified time."""
        if self.audio_duration <= 0:
            return
            
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        # Calculate the halfway point of the audio
        halfway_time = self.audio_duration / 2
        
        # Determine which timeline to use based on time
        if time_sec <= halfway_time:
            # First half timeline
            x_pos = timeline_start_x + (time_sec / halfway_time) * timeline_width
            
            # Create marker
            marker = self.timeline_canvas1.create_oval(
                x_pos-4, self.timeline_y1-4, x_pos+4, self.timeline_y1+4, 
                fill=color, outline="black", width=1
            )
            self.timeline_markers1.append(marker)
            
            # Only update progress line if in the first half
            if time_sec <= halfway_time:
                self.timeline_canvas1.coords(
                    self.progress_line1, 
                    x_pos, self.timeline_y1-15, 
                    x_pos, self.timeline_y1+15
                )
                self.timeline_canvas1.itemconfig(self.progress_line1, state="normal")
                
        else:
            # Second half timeline - adjust position to start from beginning of second timeline
            adjusted_time = time_sec - halfway_time
            x_pos = timeline_start_x + (adjusted_time / halfway_time) * timeline_width
            
            # Create marker
            marker = self.timeline_canvas2.create_oval(
                x_pos-4, self.timeline_y2-4, x_pos+4, self.timeline_y2+4, 
                fill=color, outline="black", width=1
            )
            self.timeline_markers2.append(marker)
            
            # Update progress line on second timeline
            self.timeline_canvas2.coords(
                self.progress_line2, 
                x_pos, self.timeline_y2-15, 
                x_pos, self.timeline_y2+15
            )
            self.timeline_canvas2.itemconfig(self.progress_line2, state="normal")
    
    def clear_timeline(self):
        """Clear all markers from both timelines."""
        # Clear first timeline
        for marker in self.timeline_markers1:
            self.timeline_canvas1.delete(marker)
        self.timeline_markers1 = []
        
        # Hide first progress line
        self.timeline_canvas1.itemconfig(self.progress_line1, state="hidden")
        
        # Clear second timeline
        for marker in self.timeline_markers2:
            self.timeline_canvas2.delete(marker)
        self.timeline_markers2 = []
        
        # Hide second progress line
        self.timeline_canvas2.itemconfig(self.progress_line2, state="hidden")
    
    def update_progress(self, elapsed_time):
        """Update progress bar and timeline progress lines."""
        if self.audio_duration <= 0:
            return
            
        # Update progress bar - based on full duration
        progress_percent = min(100, (elapsed_time / self.audio_duration) * 100)
        self.progress_var.set(progress_percent)
        
        # Calculate the halfway point of the audio
        halfway_time = self.audio_duration / 2
        
        # Update the appropriate timeline progress line
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        if elapsed_time <= halfway_time:
            # First half timeline
            x_pos = timeline_start_x + (elapsed_time / halfway_time) * timeline_width
            
            self.timeline_canvas1.coords(
                self.progress_line1, 
                x_pos, self.timeline_y1 - 15, 
                x_pos, self.timeline_y1 + 15
            )
            self.timeline_canvas1.itemconfig(self.progress_line1, state="normal")
            
            # Hide second timeline progress line when in first half
            self.timeline_canvas2.itemconfig(self.progress_line2, state="hidden")
        else:
            # Second half timeline
            # Keep first timeline progress line at the end
            self.timeline_canvas1.coords(
                self.progress_line1, 
                self.timeline_end_x, self.timeline_y1 - 15, 
                self.timeline_end_x, self.timeline_y1 + 15
            )
            
            # Update second timeline progress line
            adjusted_time = elapsed_time - halfway_time
            x_pos = timeline_start_x + (adjusted_time / halfway_time) * timeline_width
            
            self.timeline_canvas2.coords(
                self.progress_line2, 
                x_pos, self.timeline_y2 - 15, 
                x_pos, self.timeline_y2 + 15
            )
            self.timeline_canvas2.itemconfig(self.progress_line2, state="normal")
    
    def update_timeline_with_duration(self):
        """Update both timelines with markers based on audio duration."""
        if self.audio_duration <= 0:
            return
            
        # Clear existing time markers
        for canvas in [self.timeline_canvas1, self.timeline_canvas2]:
            for item in canvas.find_all():
                if canvas.type(item) == "text" and item != self.click_counter_text:
                    canvas.delete(item)
        
        # Draw timelines again to ensure they're clear
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        # First timeline base line
        self.timeline_canvas1.create_line(timeline_start_x, self.timeline_y1, 
                                        self.timeline_end_x, self.timeline_y1, width=2)
        
        # Second timeline base line
        self.timeline_canvas2.create_line(timeline_start_x, self.timeline_y2, 
                                        self.timeline_end_x, self.timeline_y2, width=2)
        
        # Calculate halfway point
        halfway_time = self.audio_duration / 2
        
        # Determine a suitable interval based on duration
        if halfway_time < 30:  # Less than 30 seconds per timeline
            interval = 5  # 5 second intervals
        elif halfway_time < 120:  # Less than 2 minutes per timeline
            interval = 15  # 15 second intervals
        else:
            interval = 30  # 30 second intervals
        
        # Add time markers to first timeline
        for sec in range(0, int(halfway_time) + interval, interval):
            if sec > halfway_time:
                break
                
            x_pos = timeline_start_x + (sec / halfway_time) * timeline_width
            
            # Create tick mark
            self.timeline_canvas1.create_line(x_pos, self.timeline_y1 - 5, 
                                           x_pos, self.timeline_y1 + 5, width=1)
            
            # Format time label
            if sec >= 60:
                # Show as minutes:seconds for longer intervals
                mins = sec // 60
                secs = sec % 60
                label = f"{mins}:{secs:02d}"
            else:
                # Show as seconds for shorter intervals
                label = f"{sec}s"
                
            self.timeline_canvas1.create_text(x_pos, self.timeline_y1 + 12, 
                                           text=label, font=("Arial", 8))
        
        # Add time markers to second timeline
        for sec in range(0, int(halfway_time) + interval, interval):
            if sec > halfway_time:
                break
                
            x_pos = timeline_start_x + (sec / halfway_time) * timeline_width
            
            # Calculate actual time (offset by halfway point)
            actual_sec = sec + int(halfway_time)
            
            # Create tick mark
            self.timeline_canvas2.create_line(x_pos, self.timeline_y2 - 5, 
                                           x_pos, self.timeline_y2 + 5, width=1)
            
            # Format time label
            if actual_sec >= 60:
                # Show as minutes:seconds for longer intervals
                mins = actual_sec // 60
                secs = actual_sec % 60
                label = f"{mins}:{secs:02d}"
            else:
                # Show as seconds for shorter intervals
                label = f"{actual_sec}s"
                
            self.timeline_canvas2.create_text(x_pos, self.timeline_y2 + 12, 
                                           text=label, font=("Arial", 8))
        
        # Add tactile event markers
        for t_time in self.tactile_times:
            self.add_timeline_marker(t_time, "blue")
    
    def start_experiment(self):
        """Start the experiment."""
        # Get participant ID
        try:
            self.participant_id = int(self.participant_var.get())
        except (ValueError, TypeError):
            messagebox.showerror("Error", "Please select a valid participant ID")
            return
        
        print(f"\n===== STARTING EXPERIMENT FOR PARTICIPANT {self.participant_id} =====")
        
        # Reset state
        self.stop_audio = False
        self.mouse_clicks = []
        self.click_count = 0
        self.experiment_running = True
        self.clear_timeline()
        
        # Update status
        self.status_var.set(f"Starting experiment for participant {self.participant_id}...")
        
        # Update UI
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        self.click_canvas.itemconfig(self.click_counter_text, text=f"Clicks: 0")
        
        # Run experiment in background thread
        threading.Thread(target=self.run_experiment, daemon=True).start()
    
    def stop_experiment(self):
        """Stop the experiment."""
        if not self.experiment_running:
            return
            
        self.stop_audio = True
        sd.stop()
        print("Experiment stopped manually")
        
        # Update status
        self.status_var.set("Experiment stopped")
        
        # Update UI
        self.start_button.config(state=tk.NORMAL)
        self.stop_button.config(state=tk.DISABLED)
        
        # Save click data
        self.save_click_data()
    
    def run_experiment(self):
        """Run the experiment (play audio files and track responses)."""
        try:
            # Get audio file paths
            looming_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{self.participant_id}_design_looming.wav")
            tactile_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{self.participant_id}_design_tactile.wav")
            
            print(f"Using audio files:")
            print(f"- Looming: {looming_file}")
            print(f"- Tactile: {tactile_file}")
            
            # Verify files exist
            if not os.path.exists(looming_file) or not os.path.exists(tactile_file):
                self.status_var.set(f"Error: Audio files not found")
                print(f"ERROR: Missing audio files")
                self.experiment_running = False
                self.root.after(0, lambda: self.start_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                return
            
            # Load tactile stimulus times
            self.tactile_times = self.load_tactile_times(self.participant_id)
            if not self.tactile_times:
                self.status_var.set(f"Warning: No tactile stimulus times found")
                print(f"WARNING: No tactile stimulus times found")
            
            # Load audio files to get duration
            looming_info = sf.info(looming_file)
            tactile_info = sf.info(tactile_file)
            
            # Set audio duration for timeline
            self.audio_duration = looming_info.duration
            print(f"Audio duration: {self.audio_duration:.2f} seconds")
            
            # Update timeline based on duration
            self.root.after(0, self.update_timeline_with_duration)
            
            # Ensure the progress indicators are hidden and positioned at the start
            self.timeline_canvas1.coords(
                self.progress_line1,
                50, self.timeline_y1-15,
                50, self.timeline_y1+15
            )
            self.timeline_canvas1.itemconfig(self.progress_line1, state="hidden")
            
            self.timeline_canvas2.coords(
                self.progress_line2,
                50, self.timeline_y2-15,
                50, self.timeline_y2+15
            )
            self.timeline_canvas2.itemconfig(self.progress_line2, state="hidden")
            
            # Load audio data
            print("Loading audio data...")
            looming_data, looming_sr = sf.read(looming_file)
            tactile_data, tactile_sr = sf.read(tactile_file)
            
            print(f"Audio loaded - Looming: {looming_data.shape}, Tactile: {tactile_data.shape}")
            
            # Convert stereo to mono if needed
            if len(looming_data.shape) > 1 and looming_data.shape[1] > 1:
                print("Converting looming audio from stereo to mono for playback")
                looming_data = np.mean(looming_data, axis=1)
            if len(tactile_data.shape) > 1 and tactile_data.shape[1] > 1:
                print("Converting tactile audio from stereo to mono for playback")
                tactile_data = np.mean(tactile_data, axis=1)
            
            # Set start time just before playback
            self.start_time = time.perf_counter()
            print(f"Starting audio playback at {datetime.datetime.now().strftime('%H:%M:%S.%f')}")
            
            # Initialize sounddevice audio streams directly
            print("Initializing audio streams...")
            
            # Check available devices
            devices = sd.query_devices()
            print(f"Available audio devices:")
            for i, dev in enumerate(devices):
                print(f"  {i}: {dev['name']}")
            
            # Use default output device if possible
            default_device = sd.query_devices(kind='output')
            print(f"Default output device: {default_device['name']}")
            
            # Play both audio files in separate threads
            def play_looming():
                try:
                    print("Starting looming audio playback...")
                    # Try to use device 0, fall back to default if it fails
                    try:
                        stream = sd.OutputStream(samplerate=looming_sr, channels=1, device=0)
                    except:
                        print("Falling back to default output device for looming")
                        stream = sd.OutputStream(samplerate=looming_sr, channels=1)
                        
                    stream.start()
                    
                    chunk_size = 1024
                    for i in range(0, len(looming_data), chunk_size):
                        if self.stop_audio:
                            print("Looming audio playback stopped")
                            break
                        chunk = looming_data[i:min(i+chunk_size, len(looming_data))]
                        stream.write(chunk.astype(np.float32))
                    
                    stream.stop()
                    stream.close()
                    print("Looming audio playback completed")
                except Exception as e:
                    print(f"Error playing looming audio: {e}")
                    traceback.print_exc()
            
            def play_tactile():
                try:
                    print("Starting tactile audio playback...")
                    # Try to use device 1, fall back to default if it fails
                    try:
                        stream = sd.OutputStream(samplerate=tactile_sr, channels=1, device=1)
                    except:
                        print("Falling back to default output device for tactile")
                        stream = sd.OutputStream(samplerate=tactile_sr, channels=1)
                        
                    stream.start()
                    
                    chunk_size = 1024
                    for i in range(0, len(tactile_data), chunk_size):
                        if self.stop_audio:
                            print("Tactile audio playback stopped")
                            break
                        chunk = tactile_data[i:min(i+chunk_size, len(tactile_data))]
                        stream.write(chunk.astype(np.float32))
                    
                    stream.stop()
                    stream.close()
                    print("Tactile audio playback completed")
                except Exception as e:
                    print(f"Error playing tactile audio: {e}")
                    traceback.print_exc()
            
            looming_thread = threading.Thread(target=play_looming, daemon=True)
            tactile_thread = threading.Thread(target=play_tactile, daemon=True)
            
            print("Starting audio threads...")
            looming_thread.start()
            tactile_thread.start()
            
            # Progress update loop
            self.status_var.set("Experiment running - click when you hear a tactile stimulus")
            
            # Update progress bar and timeline periodically
            update_interval = 0.1  # seconds
            end_time = self.start_time + self.audio_duration + 2.0  # Add a small buffer
            
            while (time.perf_counter() < end_time and 
                   not self.stop_audio and 
                   (looming_thread.is_alive() or tactile_thread.is_alive())):
                
                elapsed = time.perf_counter() - self.start_time
                
                # Update progress UI
                self.root.after(0, lambda t=elapsed: self.update_progress(t))
                
                # Update status with time
                minutes = int(elapsed // 60)
                seconds = int(elapsed % 60)
                total_minutes = int(self.audio_duration // 60)
                total_seconds = int(self.audio_duration % 60)
                
                self.root.after(0, lambda m=minutes, s=seconds, tm=total_minutes, ts=total_seconds: 
                              self.status_var.set(f"Experiment running - {m:02d}:{s:02d} / {tm:02d}:{ts:02d}"))
                
                time.sleep(update_interval)
            
            # Wait for threads to complete
            looming_thread.join(timeout=1.0)
            tactile_thread.join(timeout=1.0)
            
            if not self.stop_audio:
                print(f"Audio playback completed at {datetime.datetime.now().strftime('%H:%M:%S.%f')}")
                self.status_var.set("Experiment completed")
                
                # Save click data
                self.save_click_data()
            
        except Exception as e:
            print(f"ERROR during experiment: {e}")
            import traceback
            traceback.print_exc()
            self.status_var.set(f"Error: {str(e)}")
        
        finally:
            self.experiment_running = False
            
            # Update UI
            self.root.after(0, lambda: self.start_button.config(state=tk.NORMAL))
            self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
    
    def save_click_data(self):
        """Save the mouse click data to CSV."""
        if not self.mouse_clicks:
            print("No click data to save")
            return
            
        try:
            # Create DataFrame from mouse clicks
            df = pd.DataFrame(self.mouse_clicks)
            
            # Add participant ID
            df['participant_id'] = self.participant_id
            
            # Save to CSV
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = os.path.join(RESULTS_DIR, f"participant_{self.participant_id}_clicks_{timestamp}.csv")
            
            df.to_csv(filename, index=False)
            print(f"Saved click data to {filename}")
            
            # Show success message
            self.status_var.set(f"Click data saved to {os.path.basename(filename)}")
            
        except Exception as e:
            print(f"Error saving click data: {e}")
            self.status_var.set(f"Error saving click data: {str(e)}")

def main():
    app = MinimalExperimentRunner()
    app.root.mainloop()

if __name__ == "__main__":
    main()

IndentationError: unindent does not match any outer indentation level (<string>, line 12)

In [11]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Minimal Experiment Runner

This is a streamlined version focusing on core functionality:
- Loading and playing the two audio files
- Displaying the correct timeline length
- Showing the tactile stimulus times
- Capturing and displaying mouse clicks

Author: AI Assistant
"""

import os
import time
import tkinter as tk
from tkinter import ttk, messagebox
import sounddevice as sd
import soundfile as sf
import numpy as np
import pandas as pd
import threading
import datetime
import re
import glob
import traceback
from pathlib import Path

# Configuration
BASE_DIR = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace"
EXPERIMENT_AUDIO_DIR = os.path.join(BASE_DIR, "Level1_AudioGeneration", "ExperimentAudio")
EXPERIMENT_LOG_DIR = os.path.join(BASE_DIR, "Level1_AudioGeneration", "ExperimentLog")
RESULTS_DIR = os.path.join(BASE_DIR, "Level2_RunExperiment", "Results")

# Ensure results directory exists
os.makedirs(RESULTS_DIR, exist_ok=True)

class MinimalExperimentRunner:
    def __init__(self):
        # Initialize variables
        self.participant_id = None
        self.available_participants = []
        self.experiment_running = False
        self.start_time = None
        self.mouse_clicks = []
        self.tactile_times = []
        self.timeline_markers1 = []
        self.timeline_markers2 = []
        self.click_count = 0
        self.audio_duration = 0  # Will be set based on loaded audio
        
        # Flag to control audio playback
        self.stop_audio = False
        
        # Scan for available participants
        self.scan_available_participants()
        
        # Create GUI
        self.create_gui()

    def scan_available_participants(self):
        """Scan for available participants based on design files."""
        self.available_participants = []
        
        # Look for participant design files
        pattern = os.path.join(EXPERIMENT_LOG_DIR, "participant_*_design.csv")
        
        for file_path in glob.glob(pattern):
            # Extract participant ID from filename
            match = re.search(r'participant_(\d+)_design\.csv', os.path.basename(file_path))
            if match:
                participant_id = int(match.group(1))
                
                # Check if corresponding audio files exist
                looming_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{participant_id}_design_looming.wav")
                tactile_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{participant_id}_design_tactile.wav")
                
                if os.path.exists(looming_file) and os.path.exists(tactile_file):
                    self.available_participants.append(participant_id)
        
        # Sort numerically
        self.available_participants.sort()
        
        print(f"Found {len(self.available_participants)} available participants")
    
    def create_gui(self):
        """Create the GUI."""
        self.root = tk.Tk()
        self.root.title("PPS Experiment Runner")
        
        # Set protocol for window close
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)
        
        # Get screen dimensions
        screen_width = self.root.winfo_screenwidth()
        screen_height = self.root.winfo_screenheight()
        
        # Set window size to 80% of screen
        window_width = int(screen_width * 0.8)
        window_height = int(screen_height * 0.8)
        self.root.geometry(f"{window_width}x{window_height}")
        
        # Main frame with padding
        main_frame = ttk.Frame(self.root, padding=10)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Title
        ttk.Label(main_frame, text="PPS Experiment Runner", font=("Arial", 16, "bold")).pack(pady=10)
        
        # Control buttons frame at the TOP
        control_frame = ttk.LabelFrame(main_frame, text="Experiment Control", padding=10)
        control_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # Start button
        self.start_button = ttk.Button(
            control_frame, text="START EXPERIMENT",
            command=self.start_experiment, width=20
        )
        self.start_button.pack(side=tk.LEFT, padx=10, pady=5)
        
        # Stop button
        self.stop_button = ttk.Button(
            control_frame, text="STOP",
            command=self.stop_experiment, width=10,
            state=tk.DISABLED
        )
        self.stop_button.pack(side=tk.LEFT, padx=10, pady=5)
        
        # Quit button
        ttk.Button(
            control_frame, text="QUIT",
            command=self.on_close, width=10
        ).pack(side=tk.RIGHT, padx=10, pady=5)
        
        # Participant selection frame
        participant_frame = ttk.LabelFrame(main_frame, text="Participant Selection", padding=10)
        participant_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # Horizontal frame for participant selection
        selection_frame = ttk.Frame(participant_frame)
        selection_frame.pack(fill=tk.X, pady=5)
        
        # Participant ID selection
        ttk.Label(selection_frame, text="Participant ID:").pack(side=tk.LEFT, padx=5)
        
        self.participant_var = tk.StringVar()
        if self.available_participants:
            self.participant_var.set(str(self.available_participants[0]))
        
        participant_dropdown = ttk.Combobox(
            selection_frame, 
            textvariable=self.participant_var,
            values=[str(p) for p in self.available_participants],
            width=5
        )
        participant_dropdown.pack(side=tk.LEFT, padx=5)
        
        # Refresh button
        ttk.Button(
            selection_frame, 
            text="Refresh List",
            command=self.refresh_participants
        ).pack(side=tk.LEFT, padx=10)
        
        # Status display
        status_frame = ttk.Frame(main_frame)
        status_frame.pack(fill=tk.X, pady=5)
        
        self.status_var = tk.StringVar(value="Select a participant and press START EXPERIMENT")
        self.status_label = ttk.Label(status_frame, textvariable=self.status_var, 
                                     font=("Arial", 11, "bold"))
        self.status_label.pack(pady=5)
        
        # Progress bar
        self.progress_var = tk.DoubleVar(value=0.0)
        self.progress_bar = ttk.Progressbar(
            main_frame, 
            orient="horizontal", 
            length=window_width-50,
            mode="determinate",
            variable=self.progress_var
        )
        self.progress_bar.pack(fill=tk.X, padx=10, pady=5)
        
        # Split Timeline frame - two timelines for better visualization
        timeline_frame = ttk.LabelFrame(main_frame, text="Experiment Timeline", padding=10)
        timeline_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # First half timeline
        timeline_label1 = ttk.Label(timeline_frame, text="First Half:")
        timeline_label1.pack(anchor=tk.W, pady=(0, 2))
        
        self.timeline_canvas1 = tk.Canvas(timeline_frame, width=window_width-60, height=60, bg="white")
        self.timeline_canvas1.pack(fill=tk.X, pady=2)
        
        # Second half timeline
        timeline_label2 = ttk.Label(timeline_frame, text="Second Half:")
        timeline_label2.pack(anchor=tk.W, pady=(5, 2))
        
        self.timeline_canvas2 = tk.Canvas(timeline_frame, width=window_width-60, height=60, bg="white")
        self.timeline_canvas2.pack(fill=tk.X, pady=2)
        
        # Create timelines
        timeline_start_x = 50
        self.timeline_end_x = window_width - 110
        
        # First timeline
        self.timeline_y1 = 30
        self.timeline_canvas1.create_line(timeline_start_x, self.timeline_y1, self.timeline_end_x, self.timeline_y1, width=2)
        self.timeline_canvas1.create_text(timeline_start_x - 40, self.timeline_y1, text="0:00", font=("Arial", 8))
        
        # First timeline progress indicator
        self.progress_line1 = self.timeline_canvas1.create_line(
            timeline_start_x, self.timeline_y1 - 15, timeline_start_x, self.timeline_y1 + 15, 
            width=3, fill="green", state="hidden"
        )
        
        # Second timeline
        self.timeline_y2 = 30
        self.timeline_canvas2.create_line(timeline_start_x, self.timeline_y2, self.timeline_end_x, self.timeline_y2, width=2)
        
        # Second timeline progress indicator
        self.progress_line2 = self.timeline_canvas2.create_line(
            timeline_start_x, self.timeline_y2 - 15, timeline_start_x, self.timeline_y2 + 15, 
            width=3, fill="green", state="hidden"
        )
        
        # Initialize markers list for each timeline
        self.timeline_markers1 = []
        self.timeline_markers2 = []
        
        # Mouse click visualization area
        click_frame = ttk.LabelFrame(main_frame, text="Mouse Click Area", padding=10)
        click_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Canvas for visualizing mouse clicks
        self.click_canvas = tk.Canvas(click_frame, bg="lightyellow")
        self.click_canvas.pack(fill=tk.BOTH, expand=True, pady=5)
        
        # Add text to the click area
        self.click_canvas.create_text(
            window_width//2 - 30, 50,
            text="CLICK HERE WHEN YOU HEAR THE TACTILE STIMULUS",
            font=("Arial", 14, "bold"), fill="blue"
        )
        
        # Add click counter
        self.click_counter_text = self.click_canvas.create_text(
            100, 20,
            text="Clicks: 0",
            font=("Arial", 12), fill="black"
        )
        
        # Bind mouse clicks
        self.click_canvas.bind("<Button-1>", self.on_mouse_click)
    
    def refresh_participants(self):
        """Refresh the list of available participants."""
        previous_selection = self.participant_var.get()
        
        # Scan for available participants
        self.scan_available_participants()
        
        # Update dropdown values
        participant_dropdown = self.root.nametowidget(
            self.participant_var.winfo_pathname(self.participant_var.winfo_id())
        )
        participant_dropdown['values'] = [str(p) for p in self.available_participants]
        
        # Try to keep previous selection if it still exists
        if previous_selection in [str(p) for p in self.available_participants]:
            self.participant_var.set(previous_selection)
        elif self.available_participants:
            self.participant_var.set(str(self.available_participants[0]))
        else:
            self.participant_var.set("")
    
    def on_close(self):
        """Handle window close event."""
        if self.experiment_running:
            if messagebox.askyesno("Quit", "Experiment is running. Are you sure you want to quit?"):
                # Stop audio playback when closing the window
                self.stop_audio = True
                sd.stop()
                print("Audio playback stopped")
                self.root.destroy()
        else:
            self.root.destroy()
    
    def on_mouse_click(self, event):
        """Handle mouse click events during the experiment."""
        if not self.experiment_running or self.start_time is None:
            return
            
        # Calculate time since experiment start
        current_time = time.perf_counter() - self.start_time
        
        # Add to mouse clicks list
        self.mouse_clicks.append({
            "time": current_time,
            "timestamp": datetime.datetime.now().isoformat(),
            "x": event.x,
            "y": event.y
        })
        
        # Update click count
        self.click_count += 1
        self.click_canvas.itemconfig(
            self.click_counter_text, 
            text=f"Clicks: {self.click_count}"
        )
        
        # Create a visual indication of the click
        click_x, click_y = event.x, event.y
        circle = self.click_canvas.create_oval(
            click_x-10, click_y-10, click_x+10, click_y+10, 
            fill="red", outline="black"
        )
        
        # Fade out the circle after a short time
        self.root.after(500, lambda c=circle: self.click_canvas.delete(c))
        
        # Add click marker to timeline
        self.add_timeline_marker(current_time, "red")
        
        print(f"Mouse click at {current_time:.3f} seconds")
    
    def add_timeline_marker(self, time_sec, color):
        """Add a marker to the timeline at the specified time."""
        if self.audio_duration <= 0:
            return
            
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        # Calculate the halfway point of the audio
        halfway_time = self.audio_duration / 2
        
        # Determine which timeline to use based on time
        if time_sec <= halfway_time:
            # First half timeline
            x_pos = timeline_start_x + (time_sec / halfway_time) * timeline_width
            
            # Create marker
            marker = self.timeline_canvas1.create_oval(
                x_pos-4, self.timeline_y1-4, x_pos+4, self.timeline_y1+4, 
                fill=color, outline="black", width=1
            )
            self.timeline_markers1.append(marker)
            
            # Only update progress line if in the first half
            if time_sec <= halfway_time:
                self.timeline_canvas1.coords(
                    self.progress_line1, 
                    x_pos, self.timeline_y1-15, 
                    x_pos, self.timeline_y1+15
                )
                self.timeline_canvas1.itemconfig(self.progress_line1, state="normal")
                
        else:
            # Second half timeline - adjust position to start from beginning of second timeline
            adjusted_time = time_sec - halfway_time
            x_pos = timeline_start_x + (adjusted_time / halfway_time) * timeline_width
            
            # Create marker
            marker = self.timeline_canvas2.create_oval(
                x_pos-4, self.timeline_y2-4, x_pos+4, self.timeline_y2+4, 
                fill=color, outline="black", width=1
            )
            self.timeline_markers2.append(marker)
            
            # Update progress line on second timeline
            self.timeline_canvas2.coords(
                self.progress_line2, 
                x_pos, self.timeline_y2-15, 
                x_pos, self.timeline_y2+15
            )
            self.timeline_canvas2.itemconfig(self.progress_line2, state="normal")
    
    def clear_timeline(self):
        """Clear all markers from both timelines."""
        # Clear first timeline
        for marker in self.timeline_markers1:
            self.timeline_canvas1.delete(marker)
        self.timeline_markers1 = []
        
        # Hide first progress line
        self.timeline_canvas1.itemconfig(self.progress_line1, state="hidden")
        
        # Clear second timeline
        for marker in self.timeline_markers2:
            self.timeline_canvas2.delete(marker)
        self.timeline_markers2 = []
        
        # Hide second progress line
        self.timeline_canvas2.itemconfig(self.progress_line2, state="hidden")
    
    def update_progress(self, elapsed_time):
        """Update progress bar and timeline progress lines."""
        if self.audio_duration <= 0:
            return
            
        # Update progress bar - based on full duration
        progress_percent = min(100, (elapsed_time / self.audio_duration) * 100)
        self.progress_var.set(progress_percent)
        
        # Calculate the halfway point of the audio
        halfway_time = self.audio_duration / 2
        
        # Update the appropriate timeline progress line
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        if elapsed_time <= halfway_time:
            # First half timeline
            x_pos = timeline_start_x + (elapsed_time / halfway_time) * timeline_width
            
            self.timeline_canvas1.coords(
                self.progress_line1, 
                x_pos, self.timeline_y1 - 15, 
                x_pos, self.timeline_y1 + 15
            )
            self.timeline_canvas1.itemconfig(self.progress_line1, state="normal")
            
            # Hide second timeline progress line when in first half
            self.timeline_canvas2.itemconfig(self.progress_line2, state="hidden")
        else:
            # Second half timeline
            # Keep first timeline progress line at the end
            self.timeline_canvas1.coords(
                self.progress_line1, 
                self.timeline_end_x, self.timeline_y1 - 15, 
                self.timeline_end_x, self.timeline_y1 + 15
            )
            
            # Update second timeline progress line
            adjusted_time = elapsed_time - halfway_time
            x_pos = timeline_start_x + (adjusted_time / halfway_time) * timeline_width
            
            self.timeline_canvas2.coords(
                self.progress_line2, 
                x_pos, self.timeline_y2 - 15, 
                x_pos, self.timeline_y2 + 15
            )
            self.timeline_canvas2.itemconfig(self.progress_line2, state="normal")
    
    def update_timeline_with_duration(self):
        """Update both timelines with markers based on audio duration."""
        if self.audio_duration <= 0:
            return
            
        # Clear existing time markers
        for canvas in [self.timeline_canvas1, self.timeline_canvas2]:
            for item in canvas.find_all():
                if canvas.type(item) == "text" and item != self.click_counter_text:
                    canvas.delete(item)
        
        # Draw timelines again to ensure they're clear
        timeline_start_x = 50
        timeline_width = self.timeline_end_x - timeline_start_x
        
        # First timeline base line
        self.timeline_canvas1.create_line(timeline_start_x, self.timeline_y1, 
                                        self.timeline_end_x, self.timeline_y1, width=2)
        
        # Second timeline base line
        self.timeline_canvas2.create_line(timeline_start_x, self.timeline_y2, 
                                        self.timeline_end_x, self.timeline_y2, width=2)
        
        # Calculate halfway point
        halfway_time = self.audio_duration / 2
        
        # Determine a suitable interval based on duration
        if halfway_time < 30:  # Less than 30 seconds per timeline
            interval = 5  # 5 second intervals
        elif halfway_time < 120:  # Less than 2 minutes per timeline
            interval = 15  # 15 second intervals
        else:
            interval = 30  # 30 second intervals
        
        # Add time markers to first timeline
        for sec in range(0, int(halfway_time) + interval, interval):
            if sec > halfway_time:
                break
                
            x_pos = timeline_start_x + (sec / halfway_time) * timeline_width
            
            # Create tick mark
            self.timeline_canvas1.create_line(x_pos, self.timeline_y1 - 5, 
                                           x_pos, self.timeline_y1 + 5, width=1)
            
            # Format time label
            if sec >= 60:
                # Show as minutes:seconds for longer intervals
                mins = sec // 60
                secs = sec % 60
                label = f"{mins}:{secs:02d}"
            else:
                # Show as seconds for shorter intervals
                label = f"{sec}s"
                
            self.timeline_canvas1.create_text(x_pos, self.timeline_y1 + 12, 
                                           text=label, font=("Arial", 8))
        
        # Add time markers to second timeline
        for sec in range(0, int(halfway_time) + interval, interval):
            if sec > halfway_time:
                break
                
            x_pos = timeline_start_x + (sec / halfway_time) * timeline_width
            
            # Calculate actual time (offset by halfway point)
            actual_sec = sec + int(halfway_time)
            
            # Create tick mark
            self.timeline_canvas2.create_line(x_pos, self.timeline_y2 - 5, 
                                           x_pos, self.timeline_y2 + 5, width=1)
            
            # Format time label
            if actual_sec >= 60:
                # Show as minutes:seconds for longer intervals
                mins = actual_sec // 60
                secs = actual_sec % 60
                label = f"{mins}:{secs:02d}"
            else:
                # Show as seconds for shorter intervals
                label = f"{actual_sec}s"
                
            self.timeline_canvas2.create_text(x_pos, self.timeline_y2 + 12, 
                                           text=label, font=("Arial", 8))
        
        # Add tactile event markers
        for t_time in self.tactile_times:
            self.add_timeline_marker(t_time, "blue")
    
    def parse_timestamp(self, timestamp_str):
        """Parse timestamp in format MM:SS.S to seconds."""
        if pd.isna(timestamp_str):
            return None
            
        match = re.match(r'(\d+):(\d+\.\d+)', timestamp_str)
        if match:
            minutes, seconds = match.groups()
            return float(minutes) * 60 + float(seconds)
        return None
    
    def load_tactile_times(self, participant_id):
        """Load tactile stimulus times from the design file."""
        try:
            # Load from design file
            design_file = os.path.join(EXPERIMENT_LOG_DIR, f"participant_{participant_id}_design.csv")
            print(f"Loading trial data from: {design_file}")
            
            design_df = pd.read_csv(design_file)
            print(f"Loaded design with {len(design_df)} rows")
            
            # Filter for trials with tactile stimuli (exclude catch trials)
            tactile_trials = design_df[design_df['trial_type'] != 'catch']
            
            # Extract tactile stimulus times
            tactile_times = []
            for _, row in tactile_trials.iterrows():
                if 'tactile_stimulus_timestamp' in row and pd.notna(row['tactile_stimulus_timestamp']):
                    ts_str = row['tactile_stimulus_timestamp']
                    time_sec = self.parse_timestamp(ts_str)
                    if time_sec is not None:
                        tactile_times.append(time_sec)
            
            print(f"Loaded {len(tactile_times)} tactile stimulus times")
            return tactile_times
        except Exception as e:
            print(f"Error loading tactile times: {e}")
            traceback.print_exc()
            return []
    
    def start_experiment(self):
        """Start the experiment."""
        # Get participant ID
        try:
            self.participant_id = int(self.participant_var.get())
        except (ValueError, TypeError):
            messagebox.showerror("Error", "Please select a valid participant ID")
            return
        
        print(f"\n===== STARTING EXPERIMENT FOR PARTICIPANT {self.participant_id} =====")
        
        # Reset state
        self.stop_audio = False
        self.mouse_clicks = []
        self.click_count = 0
        self.experiment_running = True
        self.clear_timeline()
        
        # Update status
        self.status_var.set(f"Starting experiment for participant {self.participant_id}...")
        
        # Update UI
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        self.click_canvas.itemconfig(self.click_counter_text, text=f"Clicks: 0")
        
        # Run experiment in background thread
        threading.Thread(target=self.run_experiment, daemon=True).start()
    
    def stop_experiment(self):
        """Stop the experiment."""
        if not self.experiment_running:
            return
            
        self.stop_audio = True
        sd.stop()
        print("Experiment stopped manually")
        
        # Update status
        self.status_var.set("Experiment stopped")
        
        # Update UI
        self.start_button.config(state=tk.NORMAL)
        self.stop_button.config(state=tk.DISABLED)
        
        # Save click data
        self.save_click_data()
    
    def run_experiment(self):
        """Run the experiment (play audio files and track responses)."""
        try:
            # Get audio file paths
            looming_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{self.participant_id}_design_looming.wav")
            tactile_file = os.path.join(EXPERIMENT_AUDIO_DIR, f"participant_{self.participant_id}_design_tactile.wav")
            
            print(f"Using audio files:")
            print(f"- Looming: {looming_file}")
            print(f"- Tactile: {tactile_file}")
            
            # Verify files exist
            if not os.path.exists(looming_file) or not os.path.exists(tactile_file):
                self.status_var.set(f"Error: Audio files not found")
                print(f"ERROR: Missing audio files")
                self.experiment_running = False
                self.root.after(0, lambda: self.start_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                return
            
            # Load tactile stimulus times
            self.tactile_times = self.load_tactile_times(self.participant_id)
            if not self.tactile_times:
                self.status_var.set(f"Warning: No tactile stimulus times found")
                print(f"WARNING: No tactile stimulus times found")
            
            # Load audio files to get duration
            looming_info = sf.info(looming_file)
            tactile_info = sf.info(tactile_file)
            
            # Set audio duration for timeline
            self.audio_duration = looming_info.duration
            print(f"Audio duration: {self.audio_duration:.2f} seconds")
            
            # Update timeline based on duration
            self.root.after(0, self.update_timeline_with_duration)
            
            # Ensure the progress indicators are hidden and positioned at the start
            self.timeline_canvas1.coords(
                self.progress_line1,
                50, self.timeline_y1-15,
                50, self.timeline_y1+15
            )
            self.timeline_canvas1.itemconfig(self.progress_line1, state="hidden")
            
            self.timeline_canvas2.coords(
                self.progress_line2,
                50, self.timeline_y2-15,
                50, self.timeline_y2+15
            )
            self.timeline_canvas2.itemconfig(self.progress_line2, state="hidden")
            
            # Load audio data
            print("Loading audio data...")
            looming_data, looming_sr = sf.read(looming_file)
            tactile_data, tactile_sr = sf.read(tactile_file)
            
            print(f"Audio loaded - Looming: {looming_data.shape}, Tactile: {tactile_data.shape}")
            
            # Convert stereo to mono if needed
            if len(looming_data.shape) > 1 and looming_data.shape[1] > 1:
                print("Converting looming audio from stereo to mono for playback")
                looming_data = np.mean(looming_data, axis=1)
            if len(tactile_data.shape) > 1 and tactile_data.shape[1] > 1:
                print("Converting tactile audio from stereo to mono for playback")
                tactile_data = np.mean(tactile_data, axis=1)
            
            # Set start time just before playback
            self.start_time = time.perf_counter()
            print(f"Starting audio playback at {datetime.datetime.now().strftime('%H:%M:%S.%f')}")
            
            # Initialize sounddevice audio streams directly
            print("Initializing audio streams...")
            
            # Check available devices
            devices = sd.query_devices()
            print(f"Available audio devices:")
            for i, dev in enumerate(devices):
                print(f"  {i}: {dev['name']}")
            
            # Use default output device if possible
            default_device = sd.query_devices(kind='output')
            print(f"Default output device: {default_device['name']}")
            
            # Play both audio files in separate threads
            def play_looming():
                try:
                    print("Starting looming audio playback...")
                    # Try to use device 0, fall back to default if it fails
                    try:
                        stream = sd.OutputStream(samplerate=looming_sr, channels=1, device=0)
                    except:
                        print("Falling back to default output device for looming")
                        stream = sd.OutputStream(samplerate=looming_sr, channels=1)
                        
                    stream.start()
                    
                    chunk_size = 1024
                    for i in range(0, len(looming_data), chunk_size):
                        if self.stop_audio:
                            print("Looming audio playback stopped")
                            break
                        chunk = looming_data[i:min(i+chunk_size, len(looming_data))]
                        stream.write(chunk.astype(np.float32))
                    
                    stream.stop()
                    stream.close()
                    print("Looming audio playback completed")
                except Exception as e:
                    print(f"Error playing looming audio: {e}")
                    traceback.print_exc()
            
            def play_tactile():
                try:
                    print("Starting tactile audio playback...")
                    # Try to use device 1, fall back to default if it fails
                    try:
                        stream = sd.OutputStream(samplerate=tactile_sr, channels=1, device=1)
                    except:
                        print("Falling back to default output device for tactile")
                        stream = sd.OutputStream(samplerate=tactile_sr, channels=1)
                        
                    stream.start()
                    
                    chunk_size = 1024
                    for i in range(0, len(tactile_data), chunk_size):
                        if self.stop_audio:
                            print("Tactile audio playback stopped")
                            break
                        chunk = tactile_data[i:min(i+chunk_size, len(tactile_data))]
                        stream.write(chunk.astype(np.float32))
                    
                    stream.stop()
                    stream.close()
                    print("Tactile audio playback completed")
                except Exception as e:
                    print(f"Error playing tactile audio: {e}")
                    traceback.print_exc()
            
            looming_thread = threading.Thread(target=play_looming, daemon=True)
            tactile_thread = threading.Thread(target=play_tactile, daemon=True)
            
            print("Starting audio threads...")
            looming_thread.start()
            tactile_thread.start()
            
            # Progress update loop
            self.status_var.set("Experiment running - click when you hear a tactile stimulus")
            
            # Update progress bar and timeline periodically
            update_interval = 0.1  # seconds
            end_time = self.start_time + self.audio_duration + 2.0  # Add a small buffer
            
            while (time.perf_counter() < end_time and 
                   not self.stop_audio and 
                   (looming_thread.is_alive() or tactile_thread.is_alive())):
                
                elapsed = time.perf_counter() - self.start_time
                
                # Update progress UI
                self.root.after(0, lambda t=elapsed: self.update_progress(t))
                
                # Update status with time
                minutes = int(elapsed // 60)
                seconds = int(elapsed % 60)
                total_minutes = int(self.audio_duration // 60)
                total_seconds = int(self.audio_duration % 60)
                
                self.root.after(0, lambda m=minutes, s=seconds, tm=total_minutes, ts=total_seconds: 
                              self.status_var.set(f"Experiment running - {m:02d}:{s:02d} / {tm:02d}:{ts:02d}"))
                
                time.sleep(update_interval)
            
            # Wait for threads to complete
            looming_thread.join(timeout=1.0)
            tactile_thread.join(timeout=1.0)
            
            if not self.stop_audio:
                print(f"Audio playback completed at {datetime.datetime.now().strftime('%H:%M:%S.%f')}")
                self.status_var.set("Experiment completed")
                
                # Save click data
                self.save_click_data()
            
        except Exception as e:
            print(f"ERROR during experiment: {e}")
            traceback.print_exc()
            self.status_var.set(f"Error: {str(e)}")
        
        finally:
            self.experiment_running = False
            
            # Update UI
            self.root.after(0, lambda: self.start_button.config(state=tk.NORMAL))
            self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
    
    def save_click_data(self):
        """Save the mouse click data to CSV."""
        if not self.mouse_clicks:
            print("No click data to save")
            return
            
        try:
            # Create DataFrame from mouse clicks
            df = pd.DataFrame(self.mouse_clicks)
            
            # Add participant ID
            df['participant_id'] = self.participant_id
            
            # Save to CSV
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = os.path.join(RESULTS_DIR, f"participant_{self.participant_id}_clicks_{timestamp}.csv")
            
            df.to_csv(filename, index=False)
            print(f"Saved click data to {filename}")
            
            # Show success message
            self.status_var.set(f"Click data saved to {os.path.basename(filename)}")
            
        except Exception as e:
            print(f"Error saving click data: {e}")
            self.status_var.set(f"Error saving click data: {str(e)}")

def main():
    app = MinimalExperimentRunner()
    app.root.mainloop()

if __name__ == "__main__":
    main()

Found 100 available participants

===== STARTING EXPERIMENT FOR PARTICIPANT 0 =====
Using audio files:
- Looming: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_looming.wav
- Tactile: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_0_design_tactile.wav
Loading trial data from: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog\participant_0_design.csv
Loaded design with 206 rows
Loaded 180 tactile stimulus times
Audio duration: 1675.26 seconds
Loading audio data...
Audio loaded - Looming: (80412672, 2), Tactile: (80412672, 2)
Converting looming audio from stereo to mono for playback
Converting tactile audio from stereo to mono for playback
Starting audio playback at 23:01:57.798758
Initializing audio streams...
Available audio devices:
  0: Microsoft Sound Mapper - Input
  1: Microphone Array (Intel® Smart 
  2: Microsoft So